# IMC 2025 - Image Matching Challenge: 3D Scene Reconstruction

**NTU SC4000 Group Project**

**Pipeline (v7 - SuperPoint + ALIKED Ensemble):**
1. EDA
2. Global feature extraction (DINOv2) -> scene clustering
3. Local features: **SuperPoint** (2048px) + **ALIKED** (1024px) ensemble
4. Dual matching: SP+LightGlue and ALIKED+LightGlue in parallel
5. RANSAC geometric verification -> match graph refinement
6. Graph-based re-clustering (connected components)
7. COLMAP incremental SfM with combined SP+ALIKED keypoints
8. Pose extraction + nearest-neighbour propagation
9. Generate submission

**v7 key changes (based on IMC2025 8th place approach):**
- **Dual detector ensemble**: SuperPoint (high-res 2048) + ALIKED (1024) via LightGlue
- Combined keypoints per image -> richer features for COLMAP
- Higher RANSAC iterations (100k) for more robust verification
- Auto-fallback to ALIKED-only if SuperPoint weights unavailable

**Metric:** mAA (mean Average Accuracy) of camera poses vs ground truth.


## Step 0: Setup & Installation

In [1]:
import subprocess, sys, importlib
from pathlib import Path

# ── Debug: show what's mounted in /kaggle/input ───────────────────────────────
print(f'Python version: {sys.version.split()[0]}')
_kaggle_input = Path('/kaggle/input')
if _kaggle_input.exists():
    _mounted = [d.name for d in sorted(_kaggle_input.iterdir()) if d.is_dir()]
    print(f'Mounted inputs: {_mounted}')

# ── Find all .whl files anywhere under /kaggle/input ─────────────────────────
def find_all_wheels():
    """Recursively collect all .whl files under /kaggle/input."""
    if not Path('/kaggle/input').exists():
        return {}
    wheel_map = {}
    for whl in Path('/kaggle/input').rglob('*.whl'):
        wheel_map[whl.stem.lower()] = whl
    if wheel_map:
        print(f'Found {len(wheel_map)} wheel(s) in /kaggle/input')
    return wheel_map

ALL_WHEELS = find_all_wheels()

# import_name → (pip_spec, required)
# kornia_moons: not actually used in this notebook → optional
PACKAGES = {
    'kornia':         ('kornia',                                                        True),
    'kornia_moons':   ('kornia-moons',                                                  False),
    'lightglue':      ('lightglue @ git+https://github.com/cvg/LightGlue.git',         True),
    'pycolmap':       ('pycolmap',                                                      True),
    'h5py':           ('h5py',                                                          True),
    'transformers':   ('transformers',                                                  True),
    'accelerate':     ('accelerate',                                                    False),
    'cv2':            ('opencv-python-headless',                                        True),
    'sklearn':        ('scikit-learn',                                                  True),
}

# Proper wheel names for files that were uploaded with shortened names
_WHEEL_RENAMES = {
    'pycolmap-cp312': 'pycolmap-3.12.5-cp312-cp312-manylinux_2_28_x86_64.whl',
}

def normalize_wheel(whl):
    """If wheel was uploaded with a non-standard name, copy to /tmp with proper name."""
    import shutil
    proper_name = _WHEEL_RENAMES.get(whl.stem)
    if proper_name:
        proper = Path('/tmp') / proper_name
        if not proper.exists():
            shutil.copy2(str(whl), str(proper))
        return proper
    return whl

def find_wheel(pkg_name, wheel_map):
    """Find best matching wheel, preferring current Python version (e.g. cp312)."""
    import sys as _sys
    prefix = pkg_name.replace('-', '_').lower()
    matches = [p for stem, p in wheel_map.items()
               if stem.startswith(prefix) or stem.startswith(prefix.replace('_', '-'))]
    if not matches:
        return None
    # Prefer wheel built for the current Python version
    py_tag = f'cp{_sys.version_info.major}{_sys.version_info.minor}'
    preferred = [p for p in matches if py_tag in p.name]
    whl = sorted(preferred or matches)[-1]
    return normalize_wheel(whl)

def pip_install(import_name, spec, wheel_map):
    # Skip if already importable
    try:
        importlib.import_module(import_name)
        print(f'  ✓ {import_name:<25} [already installed]')
        return True
    except ImportError:
        pass

    # Try offline wheel (3 attempts: with deps → no-deps → ignore-python)
    whl = find_wheel(import_name, wheel_map)
    if whl:
        for extra_flags in [
            [],                                          # normal (resolves deps)
            ['--no-deps'],                               # skip deps
            ['--no-deps', '--ignore-requires-python'],  # override python version check
        ]:
            cmd = [sys.executable, '-m', 'pip', 'install', '-q', str(whl)] + extra_flags
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(f'  ✓ {import_name:<25} [wheel: {whl.name}]')
                return True
        print(f'  ✗ {import_name:<25} [wheel failed — platform mismatch: {whl.name}]')


    # Try PyPI / GitHub (only works if internet is on)
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', spec]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  ✓ {import_name:<25} [PyPI/GitHub]')
        return True
    else:
        print(f'  ✗ {import_name:<25} [FAILED — no compatible wheel & no internet]')
        print(f'    {result.stderr.strip()[-200:]}')
        return False

if ALL_WHEELS:
    print(f'Offline mode — {len(ALL_WHEELS)} wheels found')
else:
    print('No wheel datasets found — attempting online install (needs internet)')

failed = []
for import_name, (spec, required) in PACKAGES.items():
    ok = pip_install(import_name, spec, ALL_WHEELS)
    if not ok and required:
        failed.append(import_name)

if failed:
    print(f'\n⚠ Required packages missing: {failed}')
    print('  → Add wheel datasets via: Kaggle notebook → + Add Input → Datasets')
else:
    print('\nAll packages ready!')


Python version: 3.12.12
Mounted inputs: ['competitions', 'datasets', 'models']
Found 11 wheel(s) in /kaggle/input
Offline mode — 11 wheels found
  ✓ kornia                    [already installed]
  ✓ kornia_moons              [wheel: kornia_moons-0.2.9-py3-none-any.whl]
  ✓ lightglue                 [wheel: lightglue-0.0-py3-none-any.whl]
  ✓ pycolmap                  [wheel: pycolmap-3.12.5-cp312-cp312-manylinux_2_28_x86_64.whl]
  ✓ h5py                      [already installed]
  ✓ transformers              [already installed]
  ✓ accelerate                [already installed]
  ✓ cv2                       [already installed]
  ✓ sklearn                   [already installed]

All packages ready!


## Core Imports & Configuration

In [2]:
import os
import gc
import time
import shutil
import warnings
import itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
import h5py
from PIL import Image

import torch
import torch.nn.functional as F
import torchvision.transforms as T

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import normalize
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score, silhouette_score
)
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ── Device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
from pathlib import Path
import os

def find_data_root():
    """Auto-detect competition data path on Kaggle or locally."""
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        search_dirs = list(kaggle_input.iterdir())
        comp_dir = kaggle_input / 'competitions'
        if comp_dir.exists():
            search_dirs += list(comp_dir.iterdir())
        for d in sorted(search_dirs):
            if d.is_dir() and (d / 'train').exists() and (d / 'test').exists():
                return d, True
    local = Path(r'C:\Users\arunk\Documents\Arun works\UNI WORKS\Y3S2\image-matching-challenge-2025')
    if local.exists():
        return local, False
    rel = Path('image-matching-challenge-2025')
    if rel.exists():
        return rel, False
    return None, False

DATA_ROOT, IS_KAGGLE = find_data_root()

if DATA_ROOT is None:
    avail = []
    if Path('/kaggle/input').exists():
        avail = [d.name for d in Path('/kaggle/input').iterdir()]
    raise FileNotFoundError(
        f"Competition data not found!\n"
        f"Available in /kaggle/input: {avail}\n"
        "Fix: Right panel -> Add Input -> Competitions -> image-matching-challenge-2025"
    )

TRAIN_DIR        = DATA_ROOT / 'train'
TEST_DIR         = DATA_ROOT / 'test'
TRAIN_LABELS_CSV = DATA_ROOT / 'train_labels.csv'
TRAIN_THRESH_CSV = DATA_ROOT / 'train_thresholds.csv'

CACHE_DIR  = Path('cache');  CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('output'); OUTPUT_DIR.mkdir(exist_ok=True)

SKIP_TRAIN = IS_KAGGLE

# Offline dataset paths
WHEEL_DIR    = Path('/kaggle/input/imc2024-packages-pycolmap-lightglue-rerun-kornia')

def _find_dino_path():
    candidates = [
        Path('/kaggle/input/dinov2/pytorch/base/1'),
        Path('/kaggle/input/models/metaresearch/dinov2/pytorch/base/1'),
    ]
    for cfg in Path('/kaggle/input').rglob('config.json'):
        if 'dinov2' in str(cfg).lower():
            candidates.insert(0, cfg.parent)
    for p in candidates:
        if p.exists():
            return p
    return Path('/kaggle/input/dinov2/pytorch/base/1')
DINO_OFFLINE = _find_dino_path()

def _find_lightglue_dir():
    for p in Path('/kaggle/input').rglob('aliked-n16.pth'):
        return p.parent.parent
    for p in Path('/kaggle/input').rglob('lightglue*.whl'):
        return p.parent
    return Path('/kaggle/input/imc2024-packages-pycolmap-lightglue-rerun-kornia')
LIGHTGLUE_OFFLINE = _find_lightglue_dir()

# ═══════════════════════════════════════════════════════════════════════════════
# v7 — SuperPoint + ALIKED Ensemble (based on IMC2025 8th place approach)
# ═══════════════════════════════════════════════════════════════════════════════

# Feature mode: 'ensemble' (SP+ALIKED), 'aliked', or 'superpoint'
FEATURE_TYPE = 'ensemble'

# SuperPoint settings (high resolution for structured features)
SP_IMAGE_SIZE     = 2048
SP_MAX_KEYPOINTS  = 4096

# ALIKED settings (moderate resolution for complementary features)
ALIKED_IMAGE_SIZE = 1024
ALIKED_MAX_KEYPOINTS = 4096

# Legacy single-detector settings (used when FEATURE_TYPE != 'ensemble')
MAX_KEYPOINTS = 8192
IMAGE_SIZE    = 1696

# DINOv2
DINO_MODEL  = 'facebook/dinov2-base'
DINO_BATCH  = 32

# Pair selection
TOP_K_PAIRS = 80

# Matching thresholds
MIN_MATCHES = 15
MIN_INLIERS = 15    # higher = cleaner matches for COLMAP

# RANSAC
RANSAC_THRESH = 1.5  # tight threshold for high-quality inliers
RANSAC_ITERS  = 100000

# Submission
SUBMISSION_PATH = Path('/kaggle/working/submission.csv') if IS_KAGGLE else Path('submission.csv')

print(f'Data root      : {DATA_ROOT}')
print(f'IS_KAGGLE      : {IS_KAGGLE}')
print(f'SKIP_TRAIN     : {SKIP_TRAIN}')
print(f'Feature mode   : {FEATURE_TYPE}')
if FEATURE_TYPE == 'ensemble':
    print(f'  SP:    size={SP_IMAGE_SIZE}  kpts={SP_MAX_KEYPOINTS}')
    print(f'  ALIKED: size={ALIKED_IMAGE_SIZE}  kpts={ALIKED_MAX_KEYPOINTS}')
else:
    print(f'  size={IMAGE_SIZE}  kpts={MAX_KEYPOINTS}')
print(f'MIN_INLIERS    : {MIN_INLIERS}')
print(f'RANSAC         : thresh={RANSAC_THRESH}px  iters={RANSAC_ITERS}')


Data root      : /kaggle/input/competitions/image-matching-challenge-2025
IS_KAGGLE      : True
SKIP_TRAIN     : True
Feature mode   : ensemble
  SP:    size=2048  kpts=4096
  ALIKED: size=1024  kpts=4096
MIN_INLIERS    : 15
RANSAC         : thresh=1.5px  iters=100000


## Step 1: Data Loading & Exploratory Data Analysis

In [4]:
# ── CSV load & threshold parse (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Load CSV files ──
    train_labels     = pd.read_csv(TRAIN_LABELS_CSV)
    train_thresholds = pd.read_csv(TRAIN_THRESH_CSV)
    
    # Parse threshold strings → lists
    train_thresholds['threshold_list'] = train_thresholds['thresholds'].apply(
        lambda s: [float(x) for x in str(s).split(';')]
    )
    
    # Build a scene→thresholds lookup dict
    scene_thresholds = {
        (row['dataset'], row['scene']): row['threshold_list']
        for _, row in train_thresholds.iterrows()
    }
    
    print('train_labels columns :', list(train_labels.columns))
    print('train_labels shape   :', train_labels.shape)
    print()
    print(train_labels.head(3).to_string())


In [5]:
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.JPG', '.PNG', '.JPEG'}

def list_dataset_images(ds_dir: Path):
    """Return sorted list of image paths inside a dataset directory."""
    return sorted([p for p in ds_dir.iterdir() if p.suffix in IMG_EXTS])

# Always initialise so test pipeline can use them
train_datasets = []
dataset_images = {}
test_datasets  = []

if not SKIP_TRAIN:
    train_datasets = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
    dataset_images = {ds: list_dataset_images(TRAIN_DIR / ds) for ds in train_datasets}
    print(f'Training datasets: {len(train_datasets)}')
    for ds in train_datasets:
        imgs = dataset_images[ds]
        n_sc = train_labels[train_labels['dataset'] == ds]['scene'].nunique()
        print(f'  {ds:<45} {len(imgs):>4} images  {n_sc} scenes')
else:
    print('SKIP_TRAIN=True — skipping train discovery')

# Always discover test datasets
test_datasets = sorted([d.name for d in TEST_DIR.iterdir() if d.is_dir()])
print(f'Test datasets: {test_datasets}')


SKIP_TRAIN=True — skipping train discovery
Test datasets: ['ETs', 'stairs']


In [6]:
# ── GT scene distribution plot (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── GT scene distribution ──
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # Images per dataset
    ds_img_counts = {ds: len(dataset_images[ds]) for ds in train_datasets}
    axes[0].barh(list(ds_img_counts.keys()), list(ds_img_counts.values()), color='steelblue')
    axes[0].set_xlabel('Number of Images')
    axes[0].set_title('Images per Training Dataset')
    
    # Scenes per dataset
    scene_counts = train_labels.groupby('dataset')['scene'].nunique().reindex(train_datasets)
    axes[1].barh(scene_counts.index, scene_counts.values, color='tomato')
    axes[1].set_xlabel('Number of Scenes')
    axes[1].set_title('Scenes per Training Dataset')
    
    plt.tight_layout()
    plt.show()
    
    # Per-scene breakdown
    print('\nDetailed scene breakdown:')
    for ds in train_datasets:
        ds_df = train_labels[train_labels['dataset'] == ds]
        scenes = ds_df.groupby('scene')['image'].count()
        print(f'\n  {ds}')
        for sc, cnt in scenes.items():
            print(f'    └─ {sc:<40} {cnt:>4} images')


In [7]:
# ── Sample image visualisation (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Visualise sample images from a few datasets ──
    def show_samples(dataset_name, n=6):
        imgs = dataset_images[dataset_name][:n]
        fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
        if len(imgs) == 1:
            axes = [axes]
        for ax, p in zip(axes, imgs):
            img = Image.open(p).convert('RGB')
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(p.name[:20], fontsize=7)
        plt.suptitle(dataset_name, fontsize=11)
        plt.tight_layout()
        plt.show()
    
    for ds in train_datasets[:4]:
        show_samples(ds, n=min(6, len(dataset_images[ds])))


In [8]:
# ── Resolution distribution (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Image resolution distribution ──
    def sample_resolutions(images, n=30):
        sizes = []
        for p in images[:n]:
            try:
                w, h = Image.open(p).size
                sizes.append((w, h))
            except Exception:
                pass
        return sizes
    
    print('Resolution samples:')
    for ds in train_datasets[:4]:
        szs = sample_resolutions(dataset_images[ds])
        if szs:
            ws, hs = zip(*szs)
            print(f'  {ds}: W={min(ws)}-{max(ws)}, H={min(hs)}-{max(hs)}')


## Step 2: Global Feature Extraction with DINOv2

DINOv2's CLS-token embedding captures global scene appearance — crucial for separating scenes before running SfM.

In [9]:
from transformers import AutoImageProcessor, AutoModel

if DINO_OFFLINE.exists():
    _dino_src = str(DINO_OFFLINE)
    print(f'Loading DINOv2 from local dataset: {DINO_OFFLINE}')
else:
    _dino_src = DINO_MODEL
    print(f'Loading DINOv2 from HuggingFace: {DINO_MODEL}')

dino_processor = AutoImageProcessor.from_pretrained(_dino_src)
dino_model     = AutoModel.from_pretrained(_dino_src).to(device)
dino_model.eval()

print(f'DINOv2 ready — embedding dim = {dino_model.config.hidden_size}')


The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading DINOv2 from local dataset: /kaggle/input/models/metaresearch/dinov2/pytorch/large/1


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

DINOv2 ready — embedding dim = 1024


In [10]:
def extract_dino_embeddings(image_paths, batch_size=DINO_BATCH, cache_h5=None):
    """
    Extract DINOv2 CLS-token embeddings for a list of image paths.
    Cached to h5py if cache_h5 is provided.
    Returns: (embeddings [N, D] float32, names [N] list of str)
    """
    if cache_h5 and Path(cache_h5).exists():
        with h5py.File(cache_h5, 'r') as f:
            embs  = f['embeddings'][:]
            names = [n.decode() for n in f['names'][:]]
        return embs, names

    all_embs, all_names = [], []
    for i in range(0, len(image_paths), batch_size):
        batch    = image_paths[i : i + batch_size]
        pil_imgs = []
        for p in batch:
            try:
                pil_imgs.append(Image.open(p).convert('RGB'))
                all_names.append(Path(p).name)
            except Exception as e:
                print(f'  [warn] {p}: {e}')
        if not pil_imgs:
            continue
        inputs = dino_processor(images=pil_imgs, return_tensors='pt').to(device)
        with torch.no_grad():
            out = dino_model(**inputs)
        cls = out.last_hidden_state[:, 0, :].cpu().float().numpy()
        all_embs.append(cls)
        if (i // batch_size) % 5 == 0:
            print(f'  DINOv2: {min(i+batch_size, len(image_paths))}/{len(image_paths)}')

    embs = np.vstack(all_embs)
    if cache_h5:
        with h5py.File(cache_h5, 'w') as f:
            f.create_dataset('embeddings', data=embs)
            f.create_dataset('names', data=[n.encode() for n in all_names])
    return embs, all_names

# ── Train extraction (skipped on Kaggle) ──
all_dino = {}
if not SKIP_TRAIN:
    for ds in train_datasets:
        print(f'\n[{ds}]')
        cache = CACHE_DIR / f'dino_{ds}.h5'
        embs, names = extract_dino_embeddings(dataset_images[ds], cache_h5=cache)
        all_dino[ds] = {'embs': embs, 'names': names}
        print(f'  → {len(names)} embeddings, dim={embs.shape[1]}')
    print('\nDINOv2 extraction complete!')
else:
    print('SKIP_TRAIN=True — skipping train DINOv2 extraction')


SKIP_TRAIN=True — skipping train DINOv2 extraction


In [11]:
# ── Cosine similarity matrix plot (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Visualise cosine similarity matrix for one dataset ──
    def plot_similarity_matrix(ds, max_show=50):
        embs  = all_dino[ds]['embs'][:max_show]
        names = all_dino[ds]['names'][:max_show]
        norm  = normalize(embs)
        sim   = norm @ norm.T
    
        # Get GT colours
        gt_map    = dict(zip(train_labels[train_labels['dataset'] == ds]['image'],
                             train_labels[train_labels['dataset'] == ds]['scene']))
        unique_sc = sorted(set(gt_map.values()))
        sc_idx    = {s: i for i, s in enumerate(unique_sc)}
    
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
        im = axes[0].imshow(sim, vmin=0, vmax=1, cmap='hot')
        axes[0].set_title(f'{ds} – DINOv2 cosine similarity')
        plt.colorbar(im, ax=axes[0])
    
        # GT scene colours on the side
        colours = plt.cm.tab10(np.arange(len(unique_sc)) / max(len(unique_sc) - 1, 1))
        gt_bar  = np.array([sc_idx.get(gt_map.get(n, ''), -1) for n in names])
        axes[1].imshow(sim, vmin=0, vmax=1, cmap='hot')
        for k, sc in enumerate(unique_sc):
            idxs = [i for i, n in enumerate(names) if gt_map.get(n) == sc]
            axes[1].scatter([], [], c=[colours[k]], label=sc, s=50)
        axes[1].set_title('Rows coloured by GT scene')
        axes[1].legend(loc='upper right', fontsize=7)
    
        plt.tight_layout()
        plt.show()
    
    plot_similarity_matrix('ETs')
    plot_similarity_matrix('stairs')


## Step 3: Scene Clustering

For each dataset, cluster images into scenes using DINOv2 embeddings.  
During training we know `n_scenes` from the labels; for test we estimate it.

In [12]:
# ── Clustering helpers ──

def cluster_agglomerative(embs, n_clusters):
    norm = normalize(embs)
    return AgglomerativeClustering(
        n_clusters=n_clusters, metric='cosine', linkage='average'
    ).fit_predict(norm)


def cluster_kmeans(embs, n_clusters):
    norm = normalize(embs)
    return KMeans(
        n_clusters=n_clusters, random_state=42, n_init=10, max_iter=500
    ).fit_predict(norm)


def cluster_dbscan(embs, eps=0.25, min_samples=3):
    norm = normalize(embs)
    return DBSCAN(
        eps=eps, min_samples=min_samples, metric='cosine', n_jobs=-1
    ).fit_predict(norm)


def estimate_n_clusters(embs, k_min=2, k_max=12):
    """Estimate n_clusters via silhouette score (used for test data)."""
    norm = normalize(embs)
    if len(embs) < k_min * 2:
        return 1
    best_k, best_s = k_min, -1.0
    for k in range(k_min, min(k_max + 1, len(embs) // 2 + 1)):
        lbl = KMeans(n_clusters=k, random_state=42, n_init=5).fit_predict(norm)
        if len(set(lbl)) > 1:
            s = silhouette_score(norm, lbl, metric='cosine')
            if s > best_s:
                best_s, best_k = s, k
    print(f'  silhouette best k={best_k} (score={best_s:.3f})')
    return best_k


def get_gt_n_scenes(dataset_name):
    """Return number of scenes from GT labels (ignores outlier-only scenes for counting)."""
    # Count scenes that appear in train_thresholds (evaluated scenes)
    thresh_scenes = [
        (d, s) for (d, s) in scene_thresholds.keys() if d == dataset_name
    ]
    if thresh_scenes:
        return len(thresh_scenes)
    return train_labels[train_labels['dataset'] == dataset_name]['scene'].nunique()


def evaluate_clustering(pred_labels, gt_labels, valid_mask):
    """Compute ARI and NMI for the subset with known GT labels."""
    pl = pred_labels[valid_mask]
    gl = gt_labels[valid_mask]
    if len(set(pl)) < 2 or len(set(gl)) < 2:
        return 0.0, 0.0
    return adjusted_rand_score(gl, pl), normalized_mutual_info_score(gl, pl)


print('Clustering helpers defined.')

Clustering helpers defined.


In [13]:
# ── Clustering on all train datasets (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Run clustering on all training datasets ──
    
    clustering_results = {}   # dataset → dict
    
    method_scores = []   # for summary table
    
    for ds in train_datasets:
        embs  = all_dino[ds]['embs']
        names = all_dino[ds]['names']
        n_sc  = get_gt_n_scenes(ds)
    
        print(f'\n[{ds}]  n_images={len(names)}  n_scenes(GT)={n_sc}')
    
        # Build GT label array (aligned to 'names')
        gt_map  = dict(zip(
            train_labels[train_labels['dataset'] == ds]['image'],
            train_labels[train_labels['dataset'] == ds]['scene']
        ))
        unique_sc = sorted(set(gt_map.values()))
        sc2int    = {s: i for i, s in enumerate(unique_sc)}
        gt_labels = np.array([sc2int.get(gt_map.get(n, '__unknown__'), -1) for n in names])
        valid     = gt_labels >= 0
    
        results_ds = {}
    
        for method, fn in [
            ('agglomerative', lambda e: cluster_agglomerative(e, n_sc)),
            ('kmeans',        lambda e: cluster_kmeans(e, n_sc)),
        ]:
            pred = fn(embs)
            ari, nmi = evaluate_clustering(pred, gt_labels, valid)
            results_ds[method] = {'labels': pred, 'ari': ari, 'nmi': nmi}
            method_scores.append({'dataset': ds, 'method': method, 'ARI': ari, 'NMI': nmi})
            print(f'  {method:<18} ARI={ari:.3f}  NMI={nmi:.3f}')
    
        # DBSCAN (no n_clusters needed)
        pred_db = cluster_dbscan(embs, eps=0.30, min_samples=3)
        n_found = len(set(pred_db) - {-1})
        ari_db, nmi_db = evaluate_clustering(pred_db, gt_labels, valid)
        results_ds['dbscan'] = {'labels': pred_db, 'ari': ari_db, 'nmi': nmi_db}
        method_scores.append({'dataset': ds, 'method': 'dbscan', 'ARI': ari_db, 'NMI': nmi_db})
        print(f'  dbscan             ARI={ari_db:.3f}  NMI={nmi_db:.3f}  (found {n_found} clusters, {(pred_db==-1).sum()} outliers)')
    
        # Pick best method for downstream
        best_method = max(results_ds.keys(), key=lambda m: results_ds[m]['ari'])
        best_labels = results_ds[best_method]['labels']
    
        # If DBSCAN produced outliers (-1), reassign them to nearest cluster
        if -1 in best_labels:
            norm_embs = normalize(embs)
            cluster_centres = {}
            for cl in set(best_labels) - {-1}:
                cluster_centres[cl] = norm_embs[best_labels == cl].mean(0)
            for idx, lbl in enumerate(best_labels):
                if lbl == -1:
                    sims = {cl: norm_embs[idx] @ ctr for cl, ctr in cluster_centres.items()}
                    best_labels[idx] = max(sims, key=sims.get)
    
        clustering_results[ds] = {
            'labels':     best_labels,
            'names':      names,
            'n_scenes':   n_sc,
            'gt_labels':  gt_labels,
            'gt_map':     gt_map,
            'unique_sc':  unique_sc,
            'best_method': best_method,
        }
        print(f'  → using: {best_method}')
    
    print('\nClustering done!')


In [14]:
# ── Clustering summary table (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Clustering summary table ──
    scores_df = pd.DataFrame(method_scores)
    pivot = scores_df.pivot_table(index='dataset', columns='method', values='ARI')
    print('ARI per method:')
    print(pivot.round(3).to_string())
    print()
    print('Mean ARI per method:')
    print(pivot.mean().round(3))


In [15]:
# ── Clustering UMAP/PCA plot (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Visualise clustering with UMAP or PCA ──
    from sklearn.decomposition import PCA
    
    def plot_clustering(ds):
        info   = clustering_results[ds]
        embs   = all_dino[ds]['embs']
        labels = info['labels']
    
        # Reduce to 2D with PCA
        pca  = PCA(n_components=2)
        pts  = pca.fit_transform(normalize(embs))
    
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        unique_lbl = np.unique(labels)
        cmap = plt.cm.tab10
    
        # Predicted clustering
        for k, lbl in enumerate(unique_lbl):
            mask = labels == lbl
            axes[0].scatter(pts[mask, 0], pts[mask, 1],
                            c=[cmap(k / max(len(unique_lbl) - 1, 1))],
                            label=f'cluster {lbl}', s=40, alpha=0.8)
        axes[0].set_title(f'{ds} – Predicted ({info["best_method"]})')
        axes[0].legend(fontsize=8)
    
        # GT clustering
        gt_labels = info['gt_labels']
        unique_gt = np.unique(gt_labels)
        for k, lbl in enumerate(unique_gt):
            mask = gt_labels == lbl
            sc_name = info['unique_sc'][lbl] if lbl >= 0 else 'unknown'
            axes[1].scatter(pts[mask, 0], pts[mask, 1],
                            c=[cmap(k / max(len(unique_gt) - 1, 1))],
                            label=sc_name, s=40, alpha=0.8)
        axes[1].set_title(f'{ds} – Ground Truth')
        axes[1].legend(fontsize=8)
    
        plt.suptitle(f'PCA-2D embeddings – {ds}', fontsize=12)
        plt.tight_layout()
        plt.show()
    
    for ds in ['ETs', 'stairs']:
        plot_clustering(ds)


## Step 4: Local Feature Extraction (ALIKED + LightGlue)

Extract keypoints and descriptors for all images. Results are cached to h5py files.

In [16]:
from lightglue import LightGlue, ALIKED, SuperPoint
from lightglue.utils import load_image
import os, torch, shutil

# ── Copy ALL weight files (.pth, .ckpt) to torch hub cache ──────────────────
# torch.hub checks this cache before downloading. Pre-caching lets us run offline.
_HUB_DIR = Path('/kaggle/working/torch_hub')
(_HUB_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(_HUB_DIR))

_ckpt_dir = _HUB_DIR / 'checkpoints'
for _wf in Path('/kaggle/input').rglob('*'):
    if _wf.suffix in ('.pth', '.ckpt') and _wf.is_file():
        _dest = _ckpt_dir / _wf.name
        if not _dest.exists():
            shutil.copy2(str(_wf), str(_dest))
            print(f'  [weights] copied: {_wf.name}')

_found_weights = sorted([p.name for p in _ckpt_dir.iterdir() if p.is_file()])
print(f'  [weights] {len(_found_weights)} file(s): {_found_weights}')

# ── Required weight files for each mode ──────────────────────────────────────
_REQUIRED = {
    'superpoint': ['superpoint_v1.pth', 'superpoint_lightglue_v0-1_arxiv.pth'],
    'aliked':     ['aliked-n16.pth', 'aliked_lightglue_v0-1_arxiv.pth'],
}

def _check_weights(detector_name):
    missing = [f for f in _REQUIRED[detector_name] if not (_ckpt_dir / f).exists()]
    if missing:
        print(f'  [WARNING] {detector_name} weights MISSING: {missing}')
        print(f'            Add a Kaggle dataset containing these files.')
        return False
    return True

# ── Initialise models ─────────────────────────────────────────────────────────
sp_extractor  = None
sp_matcher    = None
al_extractor  = None
al_matcher    = None
ENSEMBLE_AVAILABLE = False

if FEATURE_TYPE == 'ensemble':
    # Try SuperPoint
    sp_ok = _check_weights('superpoint')
    al_ok = _check_weights('aliked')

    if al_ok:
        al_extractor = ALIKED(max_num_keypoints=ALIKED_MAX_KEYPOINTS).eval().to(device)
        al_matcher   = LightGlue(features='aliked').eval().to(device)
        print('  ALIKED + LightGlue: OK')

    if sp_ok:
        sp_extractor = SuperPoint(max_num_keypoints=SP_MAX_KEYPOINTS).eval().to(device)
        sp_matcher   = LightGlue(features='superpoint').eval().to(device)
        print('  SuperPoint + LightGlue: OK')

    if sp_ok and al_ok:
        ENSEMBLE_AVAILABLE = True
        print('Using ENSEMBLE mode (SuperPoint + ALIKED)')
    elif al_ok:
        ENSEMBLE_AVAILABLE = False
        print('SuperPoint weights missing — falling back to ALIKED-only')
    else:
        raise RuntimeError('Neither SuperPoint nor ALIKED weights found!')

elif FEATURE_TYPE == 'aliked':
    al_extractor = ALIKED(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)
    al_matcher   = LightGlue(features='aliked').eval().to(device)
    print('Using ALIKED + LightGlue (single mode)')

elif FEATURE_TYPE == 'superpoint':
    sp_extractor = SuperPoint(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)
    sp_matcher   = LightGlue(features='superpoint').eval().to(device)
    print('Using SuperPoint + LightGlue (single mode)')

# Backward-compatible aliases used by some cells
extractor = al_extractor or sp_extractor
matcher   = al_matcher or sp_matcher

print(f'Model(s) ready on {device}')
if FEATURE_TYPE == 'ensemble' and not ENSEMBLE_AVAILABLE:
    print('NOTE: Running ALIKED-only. To enable ensemble:')
    print('  Add dataset with superpoint_v1.pth + superpoint_lightglue_v0-1_arxiv.pth')


  [weights] copied: superpoint_lightglue.pth
  [weights] copied: superpoint_v1.pth
  [weights] copied: outdoor_ds.ckpt
  [weights] copied: outdoor_ot.ckpt
  [weights] copied: loftr_outdoor.ckpt
  [weights] copied: aliked_lightglue_v0-1_arxiv.pth
  [weights] copied: aliked-n16.pth
  [weights] 7 file(s): ['aliked-n16.pth', 'aliked_lightglue_v0-1_arxiv.pth', 'loftr_outdoor.ckpt', 'outdoor_ds.ckpt', 'outdoor_ot.ckpt', 'superpoint_lightglue.pth', 'superpoint_v1.pth']
  [WARNING] superpoint weights MISSING: ['superpoint_lightglue_v0-1_arxiv.pth']
            Add a Kaggle dataset containing these files.
  ALIKED + LightGlue: OK
SuperPoint weights missing — falling back to ALIKED-only
Model(s) ready on cuda
NOTE: Running ALIKED-only. To enable ensemble:
  Add dataset with superpoint_v1.pth + superpoint_lightglue_v0-1_arxiv.pth


In [17]:
def load_image_tensor(path, max_size=IMAGE_SIZE):
    """Load RGB image as float32 tensor [3,H,W] in [0,1]. EXIF-corrected."""
    from PIL import Image as PilImage, ImageOps
    import torchvision.transforms.functional as TF
    pil_img = PilImage.open(str(path)).convert('RGB')
    pil_img = ImageOps.exif_transpose(pil_img)
    img = TF.to_tensor(pil_img)
    h, w = img.shape[1], img.shape[2]
    scale = max_size / max(h, w)
    if scale < 1.0:
        nh, nw = int(h * scale), int(w * scale)
        img = F.interpolate(
            img.unsqueeze(0), size=(nh, nw), mode='bilinear', align_corners=False
        ).squeeze(0)
    return img


def _extract_to_h5(extractor_model, image_paths, cache_path, max_size, label=''):
    """Extract features with a given extractor and cache to HDF5."""
    if Path(cache_path).exists():
        print(f'  [cache hit] {cache_path}')
        return
    print(f'  Extracting {label} features for {len(image_paths)} images (size={max_size})...')
    with h5py.File(cache_path, 'w') as f:
        for i, p in enumerate(image_paths):
            name = Path(p).name
            try:
                img = load_image_tensor(p, max_size).unsqueeze(0).to(device)
                with torch.no_grad():
                    feats = extractor_model.extract(img)
                kp   = feats['keypoints'][0].cpu().numpy()
                desc = feats['descriptors'][0].cpu().numpy()
                sz   = feats.get('image_size', torch.tensor([[img.shape[3], img.shape[2]]]))
                if isinstance(sz, torch.Tensor):
                    sz = sz[0].cpu().numpy()
            except Exception as e:
                print(f'  [warn] {name}: {e}')
                kp   = np.zeros((0, 2), np.float32)
                desc = np.zeros((0, 128), np.float32)
                sz   = np.array([0, 0])
            grp = f.require_group(name)
            grp.create_dataset('keypoints',   data=kp,   compression='gzip')
            grp.create_dataset('descriptors', data=desc, compression='gzip')
            grp.create_dataset('image_size',  data=sz)
            if (i + 1) % 20 == 0:
                print(f'  {i + 1}/{len(image_paths)}')
    print(f'  Saved to {cache_path}')


def extract_and_cache_features(image_paths, cache_path, max_size=IMAGE_SIZE):
    """v5-compatible wrapper. Used by single-detector modes."""
    _extract_to_h5(extractor, image_paths, cache_path, max_size)


def extract_ensemble_features(image_paths, ds_tag):
    """Extract features with both SuperPoint and ALIKED, cache separately."""
    sp_cache = CACHE_DIR / f'feats_sp_{ds_tag}.h5'
    al_cache = CACHE_DIR / f'feats_aliked_{ds_tag}.h5'
    if sp_extractor:
        _extract_to_h5(sp_extractor, image_paths, sp_cache, SP_IMAGE_SIZE, 'SuperPoint')
    if al_extractor:
        _extract_to_h5(al_extractor, image_paths, al_cache, ALIKED_IMAGE_SIZE, 'ALIKED')
    return sp_cache, al_cache


def read_features(cache_path, name):
    """Read features for a single image from HDF5 cache."""
    with h5py.File(cache_path, 'r') as f:
        if name not in f:
            return None
        return {k: f[name][k][:] for k in f[name]}


# ── Run extraction for all training datasets (skipped on Kaggle) ──
if not SKIP_TRAIN:
    if FEATURE_TYPE == 'ensemble' and ENSEMBLE_AVAILABLE:
        for ds in train_datasets:
            print(f'\n[{ds}]')
            extract_ensemble_features(dataset_images[ds], ds)
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    else:
        for ds in train_datasets:
            print(f'\n[{ds}]')
            cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_{ds}.h5'
            extract_and_cache_features(dataset_images[ds], cache)
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    print('\nLocal feature extraction complete!')


## Step 5: Image Pair Selection & Feature Matching

1. **Coarse**: Select top-K nearest neighbours per image using DINOv2 cosine similarity (within predicted scene cluster).
2. **Fine**: Match selected pairs with LightGlue.
3. **Verify**: Apply RANSAC on fundamental matrix to remove outlier matches.
4. **Refine**: Use match counts to refine scene clustering via connected components.

In [18]:
def select_pairs(embs, names, cluster_labels, top_k=TOP_K_PAIRS):
    """Smart pair selection.
    - Cluster <= 20 images: exhaustive.
    - Larger: top-K DINOv2 neighbours with >= 30% coverage guarantee."""
    EXHAUSTIVE_THRESH = 20
    norm = normalize(embs)
    sim  = norm @ norm.T
    np.fill_diagonal(sim, -1)
    pairs = set()
    for cl_id in np.unique(cluster_labels):
        cl_idx = np.where(cluster_labels == cl_id)[0]
        n_cl   = len(cl_idx)
        if n_cl <= EXHAUSTIVE_THRESH:
            for a in range(n_cl):
                for b in range(a + 1, n_cl):
                    pairs.add((min(cl_idx[a], cl_idx[b]),
                               max(cl_idx[a], cl_idx[b])))
        else:
            k_eff = min(max(top_k, int(n_cl * 0.30)), n_cl - 1)
            for i in cl_idx:
                sim_i = sim[i].copy()
                sim_i[cluster_labels != cl_id] = -1
                sim_i[i] = -1
                topk = np.argsort(sim_i)[::-1][:k_eff]
                for j in topk:
                    if sim_i[j] > -1:
                        pairs.add((min(i, j), max(i, j)))
    return sorted(pairs)


def _match_lightglue(feat0, feat1, matcher_model):
    """Run a LightGlue matcher on pre-extracted features. Returns [M, 2]."""
    def to_lg(feat):
        return {
            'keypoints':   torch.tensor(feat['keypoints'],   device=device).unsqueeze(0),
            'descriptors': torch.tensor(feat['descriptors'], device=device).unsqueeze(0),
            'image_size':  torch.tensor(feat['image_size'],  device=device).unsqueeze(0),
        }
    with torch.no_grad():
        out = matcher_model({'image0': to_lg(feat0), 'image1': to_lg(feat1)})
    return out['matches'][0].cpu().numpy()


def match_pair_lightglue(feat0, feat1):
    """Backward-compatible: use the default (ALIKED) matcher."""
    return _match_lightglue(feat0, feat1, matcher)


def ransac_verify(kp0, kp1, matches):
    """RANSAC fundamental matrix filter. Returns inlier match rows."""
    if len(matches) < 8:
        return np.empty((0, 2), np.int32)
    pts0 = kp0[matches[:, 0]]
    pts1 = kp1[matches[:, 1]]
    _, mask = cv2.findFundamentalMat(
        pts0, pts1,
        method=cv2.USAC_MAGSAC,
        ransacReprojThreshold=RANSAC_THRESH,
        confidence=0.99999,
        maxIters=RANSAC_ITERS,
    )
    if mask is None:
        return np.empty((0, 2), np.int32)
    return matches[mask.ravel().astype(bool)]


def _match_one_detector(feat_cache, names, pairs, matcher_model, verbose=True, label=''):
    """Run matching with a single detector's features.
    Returns {(ni, nj): inlier_matches [M, 2]}."""
    verified = {}
    with h5py.File(feat_cache, 'r') as feat_file:
        for idx, (i, j) in enumerate(pairs):
            ni, nj = names[i], names[j]
            if ni not in feat_file or nj not in feat_file:
                continue
            f0 = {k: feat_file[ni][k][:] for k in feat_file[ni]}
            f1 = {k: feat_file[nj][k][:] for k in feat_file[nj]}
            if len(f0['keypoints']) < 10 or len(f1['keypoints']) < 10:
                continue
            try:
                matches = _match_lightglue(f0, f1, matcher_model)
                if len(matches) < MIN_MATCHES:
                    continue
                inliers = ransac_verify(f0['keypoints'], f1['keypoints'], matches)
                if len(inliers) >= MIN_INLIERS:
                    verified[(ni, nj)] = inliers
            except Exception as e:
                if verbose:
                    print(f'  [{label} warn] {ni}x{nj}: {e}')
            if verbose and (idx + 1) % 200 == 0:
                print(f'  [{label}] {idx + 1}/{len(pairs)}  ({len(verified)} verified)')
    return verified


print('Pair selection & matching helpers defined.')


Pair selection & matching helpers defined.


In [19]:
def run_matching_for_dataset(ds, verbose=True):
    """Full matching pipeline for a training dataset.

    Ensemble mode: runs both SP+LG and ALIKED+LG, returns merged result.
    Single mode: runs one detector, same as v5.

    Returns:
      verified      : {(ni,nj): dummy_array} for graph refinement (len = total inliers)
      sp_verified   : {(ni,nj): sp_inliers [M,2]} or empty
      al_verified   : {(ni,nj): al_inliers [M,2]} or empty
    """
    embs   = all_dino[ds]['embs']
    names  = all_dino[ds]['names']
    labels = clustering_results[ds]['labels']

    pairs = select_pairs(embs, names, labels, top_k=TOP_K_PAIRS)
    if verbose:
        print(f'  {len(pairs)} candidate pairs')

    sp_verified = {}
    al_verified = {}

    if FEATURE_TYPE == 'ensemble' and ENSEMBLE_AVAILABLE:
        sp_cache = CACHE_DIR / f'feats_sp_{ds}.h5'
        al_cache = CACHE_DIR / f'feats_aliked_{ds}.h5'

        if verbose:
            print(f'  --- SuperPoint pass ---')
        sp_verified = _match_one_detector(
            sp_cache, names, pairs, sp_matcher, verbose, label='SP')
        if verbose:
            print(f'  -> {len(sp_verified)} SP verified')

        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        if verbose:
            print(f'  --- ALIKED pass ---')
        al_verified = _match_one_detector(
            al_cache, names, pairs, al_matcher, verbose, label='AL')
        if verbose:
            print(f'  -> {len(al_verified)} ALIKED verified')
    else:
        # Single detector mode
        cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_{ds}.h5'
        al_verified = _match_one_detector(
            cache, names, pairs, matcher, verbose, label=FEATURE_TYPE)
        if verbose:
            print(f'  -> {len(al_verified)} verified')

    # Merge for graph refinement: combined count per pair
    all_keys = set(sp_verified.keys()) | set(al_verified.keys())
    verified = {}
    for key in all_keys:
        sp_n = len(sp_verified.get(key, []))
        al_n = len(al_verified.get(key, []))
        # Dummy array whose len() = total inlier count (for refine_clusters)
        verified[key] = np.zeros((sp_n + al_n, 2), dtype=np.int32)

    if verbose:
        print(f'  -> {len(verified)} total unique verified pairs')

    return verified, sp_verified, al_verified


# ── Run for all training datasets (skipped on Kaggle) ──
all_verified    = {}
all_sp_verified = {}
all_al_verified = {}

if not SKIP_TRAIN:
    for ds in train_datasets:
        print(f'\n[{ds}]')
        t0 = time.time()
        verified, sp_v, al_v = run_matching_for_dataset(ds)
        all_verified[ds]    = verified
        all_sp_verified[ds] = sp_v
        all_al_verified[ds] = al_v
        print(f'  done in {time.time()-t0:.1f}s')
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    print('\nMatching complete!')


In [20]:
# ── Match-graph-based clustering refinement ──
# Images connected by strong matches belong to the same scene.
# We use Union-Find to re-label scene clusters based on verified matches.

class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def refine_clusters_with_match_graph(names, verified_pairs, cluster_labels, min_edge_weight=5):
    """
    Use connected components of the verified-match graph to refine clustering.
    Two images are joined if they have >= min_edge_weight inlier matches.
    Returns new label array.
    """
    name2idx = {n: i for i, n in enumerate(names)}
    uf = UnionFind(len(names))

    for (ni, nj), inliers in verified_pairs.items():
        if len(inliers) >= min_edge_weight:
            if ni in name2idx and nj in name2idx:
                uf.union(name2idx[ni], name2idx[nj])

    # Connected component labels
    roots = [uf.find(i) for i in range(len(names))]
    unique_roots = sorted(set(roots))
    root2lbl = {r: k for k, r in enumerate(unique_roots)}
    cc_labels = np.array([root2lbl[r] for r in roots])

    # Merge small components (< 3 images) with their nearest DINOv2 cluster
    cc_sizes = np.bincount(cc_labels)
    final_labels = cc_labels.copy()
    dominant = [l for l, s in enumerate(cc_sizes) if s >= 3]

    if not dominant:
        # Fall back to DINOv2 clusters
        return cluster_labels

    for idx, lbl in enumerate(cc_labels):
        if cc_sizes[lbl] < 3:
            # Use DINOv2 cluster label
            final_labels[idx] = cluster_labels[idx]

    return final_labels


# ── Refine and re-evaluate ──
refined_results = {}

for ds in train_datasets:
    info    = clustering_results[ds]
    refined = refine_clusters_with_match_graph(
        info['names'], all_verified[ds], info['labels'], min_edge_weight=MIN_INLIERS
    )
    gt_lbl = info['gt_labels']
    valid  = gt_lbl >= 0
    ari, nmi = evaluate_clustering(refined, gt_lbl, valid)

    n_before = len(set(info['labels']))
    n_after  = len(set(refined))
    print(f'[{ds}]  clusters: {n_before}→{n_after}  ARI: {info.get("ari", 0):.3f}→{ari:.3f}')

    refined_results[ds] = {
        **info,
        'labels':     refined,
        'ari_refined': ari,
        'nmi_refined': nmi,
    }

print('\nRefinement done!')


Refinement done!


In [21]:
# ── Visualise sample matches for one pair ──
def visualise_matches(ds, pair_idx=0):
    pairs = list(all_verified[ds].items())
    if not pairs or pair_idx >= len(pairs):
        print('No verified pairs.')
        return

    (ni, nj), inliers = pairs[pair_idx]
    cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_{ds}.h5'

    with h5py.File(cache, 'r') as f:
        kp0 = f[ni]['keypoints'][:]
        kp1 = f[nj]['keypoints'][:]

    img0 = np.array(Image.open(TRAIN_DIR / ds / ni).convert('RGB'))
    img1 = np.array(Image.open(TRAIN_DIR / ds / nj).convert('RGB'))

    # Resize to same height
    h0, h1 = img0.shape[0], img1.shape[0]
    if h0 != h1:
        scale = h0 / h1
        img1  = cv2.resize(img1, (int(img1.shape[1] * scale), h0))
        kp1  *= scale

    canvas = np.concatenate([img0, img1], axis=1)
    W0 = img0.shape[1]

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(canvas)
    for m in inliers[::max(1, len(inliers)//50)]:
        x0, y0 = kp0[m[0]]
        x1, y1 = kp1[m[1]]
        ax.plot([x0, x1 + W0], [y0, y1], 'lime', linewidth=0.5, alpha=0.6)
        ax.scatter([x0, x1 + W0], [y0, y1], c='red', s=5, zorder=5)

    ax.set_title(f'{ds}  {ni} ↔ {nj}  ({len(inliers)} inliers)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if not SKIP_TRAIN:
    for ds in ['ETs', 'stairs']:
        if ds in all_verified:
            visualise_matches(ds, 0)

## Step 6: Structure-from-Motion with COLMAP (pycolmap)

For each predicted scene cluster, we:
1. Create a COLMAP SQLite database
2. Import keypoints and verified matches
3. Run incremental SfM
4. Extract world-to-camera poses from the best model

In [22]:
import pycolmap
import sqlite3
_pycol_ver = getattr(pycolmap, '__version__', 'unknown')
print(f'pycolmap version: {_pycol_ver}')
# Detect API style
_opts_test = pycolmap.IncrementalPipelineOptions()
_has_ba_refine = hasattr(_opts_test.mapper, 'ba_refine_focal_length')
_has_refine    = hasattr(_opts_test.mapper, 'refine_focal_length')
print(f'  ba_refine_focal_length attr: {_has_ba_refine}')
print(f'  refine_focal_length attr   : {_has_refine}')
del _opts_test

def get_image_wh(ds_dir, name):
    try:
        return Image.open(ds_dir / name).size
    except Exception:
        return 1000, 750

def image_ids_to_pair_id(id1, id2):
    if id1 > id2:
        id1, id2 = id2, id1
    return id1 * 2147483647 + id2

def get_focal_px(image_path, w, h):
    try:
        from PIL import Image as PilImage
        img_pil = PilImage.open(image_path)
        exif_data = img_pil.getexif()
        if exif_data:
            focal_35mm = exif_data.get(41989)
            if focal_35mm and focal_35mm > 0:
                sensor_diag = (w**2 + h**2) ** 0.5
                diag_35mm = (36**2 + 24**2) ** 0.5
                focal_px = float(focal_35mm) / diag_35mm * sensor_diag
                return focal_px, True
    except Exception:
        pass
    return float(max(w, h)) * 1.2, False

def build_colmap_db(db_path, ds_dir, image_names, feat_cache, verified_pairs):
    db_path = Path(db_path)
    if db_path.exists():
        db_path.unlink()
    conn = sqlite3.connect(str(db_path))
    c = conn.cursor()
    c.executescript('''
        CREATE TABLE IF NOT EXISTS cameras (camera_id INTEGER PRIMARY KEY, model INTEGER NOT NULL, width INTEGER NOT NULL, height INTEGER NOT NULL, params BLOB, prior_focal_length INTEGER DEFAULT 0);
        CREATE TABLE IF NOT EXISTS images (image_id INTEGER PRIMARY KEY, name TEXT NOT NULL, camera_id INTEGER NOT NULL, prior_qw REAL DEFAULT 0, prior_qx REAL DEFAULT 0, prior_qy REAL DEFAULT 0, prior_qz REAL DEFAULT 0, prior_tx REAL DEFAULT 0, prior_ty REAL DEFAULT 0, prior_tz REAL DEFAULT 0);
        CREATE TABLE IF NOT EXISTS keypoints (image_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB);
        CREATE TABLE IF NOT EXISTS descriptors (image_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB);
        CREATE TABLE IF NOT EXISTS matches (pair_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB);
        CREATE TABLE IF NOT EXISTS two_view_geometries (pair_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB, config INTEGER DEFAULT 2, F BLOB, E BLOB, H BLOB, qvec BLOB, tvec BLOB);
    ''')
    SIMPLE_RADIAL = 2
    name2id = {}
    cam_id_counter = 1
    img_id_counter = 1
    with h5py.File(feat_cache, 'r') as feat_file:
        for name in image_names:
            w, h = get_image_wh(ds_dir, name)
            focal_px, has_prior = get_focal_px(ds_dir / name, w, h)
            params = np.array([focal_px, w / 2.0, h / 2.0, 0.0], dtype=np.float64)
            prior_flag = 1 if has_prior else 0
            c.execute('INSERT INTO cameras VALUES (?,?,?,?,?,?)',
                      (cam_id_counter, SIMPLE_RADIAL, w, h, params.tobytes(), prior_flag))
            c.execute('INSERT INTO images (image_id, name, camera_id) VALUES (?,?,?)',
                      (img_id_counter, name, cam_id_counter))
            name2id[name] = img_id_counter
            if name in feat_file:
                kp = feat_file[name]['keypoints'][:].astype(np.float32)
                kp6 = np.zeros((len(kp), 6), dtype=np.float32)
                kp6[:, :2] = kp
                kp6[:, 2] = 1.0
                c.execute('INSERT INTO keypoints VALUES (?,?,?,?)',
                          (img_id_counter, len(kp), 6, kp6.tobytes()))
            else:
                c.execute('INSERT INTO keypoints VALUES (?,?,?,?)',
                          (img_id_counter, 0, 6, b''))
            cam_id_counter += 1
            img_id_counter += 1
    name_set = set(image_names)
    for (ni, nj), inliers in verified_pairs.items():
        if ni in name_set and nj in name_set and ni in name2id and nj in name2id:
            id_i, id_j = name2id[ni], name2id[nj]
            pair_id = image_ids_to_pair_id(id_i, id_j)
            m32 = inliers.astype(np.uint32)
            c.execute('INSERT OR REPLACE INTO matches VALUES (?,?,?,?)',
                      (pair_id, len(m32), 2, m32.tobytes()))
            c.execute('INSERT OR REPLACE INTO two_view_geometries (pair_id, rows, cols, data, config) VALUES (?,?,?,?,?)',
                      (pair_id, len(m32), 2, m32.tobytes(), 2))
    conn.commit()
    conn.close()
    return name2id

def _try_set(obj, attr, val):
    """Safely set an attribute — silently skips if not present in this pycolmap version."""
    try:
        setattr(obj, attr, val)
    except AttributeError:
        pass

def get_colmap_options(n_images=0):
    """
    pycolmap-version-safe option builder.
    Tested against pycolmap 3.x and 4.x (4.0.2).
    Core fix: init_min_tri_angle=1.5 (default 16) rescues close-range scenes.
    """
    opts = pycolmap.IncrementalPipelineOptions()

    # ── Critical: close-range scene fix ──────────────────────
    _try_set(opts.mapper, 'init_min_tri_angle',        1.5)
    _try_set(opts.mapper, 'init_max_error',            4.0)

    # ── Pose registration ─────────────────────────────────────
    _try_set(opts.mapper, 'abs_pose_min_num_inliers',  15)
    _try_set(opts.mapper, 'abs_pose_min_inlier_ratio', 0.20)
    _try_set(opts.mapper, 'abs_pose_max_error',        12.0)

    # ── Bundle adjustment (attr names changed in pycolmap 4.x) ─
    for ba_attr in ['ba_refine_focal_length',
                    'refine_focal_length']:
        _try_set(opts.mapper, ba_attr, True)
    _try_set(opts.mapper, 'ba_local_num_images', 6)  # int, not bool

    for ba_attr in ['ba_refine_extra_params',
                    'refine_extra_params']:
        _try_set(opts.mapper, ba_attr, True)

    _try_set(opts.mapper, 'max_extra_param',           1.0)

    # ── Filtering ─────────────────────────────────────────────
    _try_set(opts.mapper, 'filter_max_reproj_error',   4.0)
    _try_set(opts.mapper, 'filter_min_tri_angle',      1.5)

    # ── Matching ──────────────────────────────────────────────
    _try_set(opts,        'min_num_matches',            MIN_INLIERS)

    return opts

def _qvec_to_R(qw, qx, qy, qz):
    """Convert COLMAP quaternion [qw,qx,qy,qz] to 3x3 rotation matrix."""
    return np.array([
        [1-2*(qy*qy+qz*qz),   2*(qx*qy-qw*qz),   2*(qx*qz+qw*qy)],
        [2*(qx*qy+qw*qz),   1-2*(qx*qx+qz*qz),   2*(qy*qz-qw*qx)],
        [2*(qx*qz-qw*qy),     2*(qy*qz+qw*qx), 1-2*(qx*qx+qy*qy)]
    ])

def _read_images_bin(path):
    """Parse COLMAP images.bin directly — no pycolmap API needed."""
    import struct
    poses = {}
    with open(path, 'rb') as f:
        num = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num):
            struct.unpack('<I', f.read(4))          # image_id
            qw,qx,qy,qz = struct.unpack('<4d', f.read(32))
            tx,ty,tz    = struct.unpack('<3d', f.read(24))
            struct.unpack('<I', f.read(4))          # camera_id
            name_b = b''
            while True:
                c = f.read(1)
                if c == b'\x00': break
                name_b += c
            name = name_b.decode('utf-8')
            n2d = struct.unpack('<Q', f.read(8))[0]
            f.read(n2d * 24)                        # skip 2D point observations
            poses[name] = (_qvec_to_R(qw,qx,qy,qz), np.array([tx,ty,tz]))
    return poses

def _read_images_txt(path):
    """Parse COLMAP images.txt directly — no pycolmap API needed."""
    poses = {}
    with open(path) as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        p = lines[i].split()
        i += 2  # every image has a pose line + a 2D-points line
        if len(p) < 10:
            continue
        qw,qx,qy,qz = float(p[1]),float(p[2]),float(p[3]),float(p[4])
        tx,ty,tz     = float(p[5]),float(p[6]),float(p[7])
        name = p[9]
        poses[name] = (_qvec_to_R(qw,qx,qy,qz), np.array([tx,ty,tz]))
    return poses

def _best_recon_dir(sfm_out):
    """Return subdirectory of sfm_out that has the most registered images."""
    sfm_out = Path(sfm_out)
    if not sfm_out.exists():
        return None
    import struct
    best_dir, best_n = None, 0
    for sub in sorted(sfm_out.iterdir()):
        if not sub.is_dir():
            continue
        img_bin = sub / 'images.bin'
        img_txt = sub / 'images.txt'
        n = 0
        if img_bin.exists():
            try:
                with open(img_bin, 'rb') as f:
                    n = struct.unpack('<Q', f.read(8))[0]
            except Exception:
                pass
        elif img_txt.exists():
            try:
                with open(img_txt) as f:
                    n = sum(1 for l in f
                            if not l.startswith('#') and l.strip() and len(l.split()) >= 10)
            except Exception:
                pass
        if n > best_n:
            best_n, best_dir = n, sub
    return best_dir

def read_poses_from_sfm_dir(sfm_out):
    """Read {image_name: (R, t)} from best reconstruction under sfm_out.
    Reads images.bin / images.txt directly — bypasses pycolmap Python API."""
    recon_dir = _best_recon_dir(sfm_out)
    if recon_dir is None:
        return {}
    img_bin = recon_dir / 'images.bin'
    img_txt = recon_dir / 'images.txt'
    try:
        if img_bin.exists():
            poses = _read_images_bin(img_bin)
            print(f'    [disk-read] {len(poses)} poses from {recon_dir.name}/images.bin')
            return poses
        if img_txt.exists():
            poses = _read_images_txt(img_txt)
            print(f'    [disk-read] {len(poses)} poses from {recon_dir.name}/images.txt')
            return poses
    except Exception as e:
        print(f'    [disk-read] failed: {e}')
    return {}

def run_sfm(db_path, image_dir, sfm_out):
    sfm_out = Path(sfm_out)
    sfm_out.mkdir(parents=True, exist_ok=True)
    try:
        pycolmap.incremental_mapping(
            database_path=str(db_path),
            image_path=str(image_dir),
            output_path=str(sfm_out),
            options=get_colmap_options(),
        )
    except Exception as e:
        print(f'  [SfM error] {e}')
    # Always read poses from disk — pycolmap 3.x in-memory objects
    # have broken Python pose access; disk files are always correct.
    return read_poses_from_sfm_dir(sfm_out)
def extract_pose(image_obj):
    """Extract (R, t) from a pycolmap Image. Returns (None, None) on failure."""
    # pycolmap 3.x: cam_from_world is a Rigid3d
    try:
        rigid3d = image_obj.cam_from_world
        R = np.array(rigid3d.rotation.matrix())
        t = np.array(rigid3d.translation)
        if R.shape == (3, 3) and t.shape == (3,):
            return R, t
    except Exception:
        pass
    # pycolmap 0.x: rotation_matrix() / tvec
    try:
        R = np.array(image_obj.rotation_matrix())
        t = np.array(image_obj.tvec)
        if R.shape == (3, 3) and t.shape == (3,):
            return R, t
    except Exception:
        pass
    # pycolmap 0.x: qvec / tvec
    try:
        from scipy.spatial.transform import Rotation as _Rot
        qvec = np.array(image_obj.qvec)  # [qw, qx, qy, qz]
        tvec = np.array(image_obj.tvec)
        R = _Rot.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]]).as_matrix()
        if R.shape == (3, 3) and tvec.shape == (3,):
            return R, tvec
    except Exception:
        pass
    return None, None  # caller must handle: do NOT fall back to identity here



# ─────────────────────────────────────────────────────────────────────────────
# build_colmap_db_ensemble — combine SP + ALIKED keypoints and matches
#
# For each image: kp_combined = vstack([sp_kp, al_kp])
# SP match indices: as-is (0..N_sp-1)
# ALIKED match indices: offset by N_sp for each image
# ─────────────────────────────────────────────────────────────────────────────
def build_colmap_db_ensemble(db_path, ds_dir, image_names,
                              sp_feat_cache, al_feat_cache,
                              sp_verified, al_verified):
    """Build COLMAP DB from SuperPoint + ALIKED ensemble features.

    Each image gets combined keypoints from both detectors.
    Match indices from ALIKED are offset by the SP keypoint count.
    """
    db_path = Path(db_path)
    if db_path.exists():
        db_path.unlink()
    conn = sqlite3.connect(str(db_path))
    c = conn.cursor()
    c.executescript(
        "CREATE TABLE IF NOT EXISTS cameras ("
        "camera_id INTEGER PRIMARY KEY, model INTEGER NOT NULL, "
        "width INTEGER NOT NULL, height INTEGER NOT NULL, "
        "params BLOB, prior_focal_length INTEGER DEFAULT 0);"
        "CREATE TABLE IF NOT EXISTS images ("
        "image_id INTEGER PRIMARY KEY, name TEXT NOT NULL, camera_id INTEGER NOT NULL, "
        "prior_qw REAL DEFAULT 0, prior_qx REAL DEFAULT 0, prior_qy REAL DEFAULT 0, "
        "prior_qz REAL DEFAULT 0, prior_tx REAL DEFAULT 0, prior_ty REAL DEFAULT 0, "
        "prior_tz REAL DEFAULT 0);"
        "CREATE TABLE IF NOT EXISTS keypoints ("
        "image_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB);"
        "CREATE TABLE IF NOT EXISTS descriptors ("
        "image_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB);"
        "CREATE TABLE IF NOT EXISTS matches ("
        "pair_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB);"
        "CREATE TABLE IF NOT EXISTS two_view_geometries ("
        "pair_id INTEGER PRIMARY KEY, rows INTEGER NOT NULL, cols INTEGER NOT NULL, data BLOB, "
        "config INTEGER DEFAULT 2, F BLOB, E BLOB, H BLOB, qvec BLOB, tvec BLOB);"
    )

    SIMPLE_RADIAL = 2
    name_set = set(image_names)
    name2id  = {}
    sp_kp_counts = {}  # name -> N_sp (offset for ALIKED indices)

    # ── Read feature caches ──────────────────────────────────────────────────
    sp_file = h5py.File(sp_feat_cache, 'r') if Path(sp_feat_cache).exists() else None
    al_file = h5py.File(al_feat_cache, 'r') if Path(al_feat_cache).exists() else None

    cam_id_counter = 1
    img_id_counter = 1

    for name in image_names:
        w, h = get_image_wh(ds_dir, name)
        focal_px, has_prior = get_focal_px(ds_dir / name, w, h)
        params = np.array([focal_px, w / 2.0, h / 2.0, 0.0], dtype=np.float64)

        c.execute('INSERT INTO cameras VALUES (?,?,?,?,?,?)',
                  (cam_id_counter, SIMPLE_RADIAL, w, h,
                   params.tobytes(), 1 if has_prior else 0))
        c.execute('INSERT INTO images (image_id, name, camera_id) VALUES (?,?,?)',
                  (img_id_counter, name, cam_id_counter))
        name2id[name] = img_id_counter

        # Combine keypoints from both detectors
        sp_kp = np.zeros((0, 2), np.float32)
        al_kp = np.zeros((0, 2), np.float32)
        if sp_file and name in sp_file:
            sp_kp = sp_file[name]['keypoints'][:].astype(np.float32)
        if al_file and name in al_file:
            al_kp = al_file[name]['keypoints'][:].astype(np.float32)

        sp_kp_counts[name] = len(sp_kp)
        combined_kp = np.vstack([sp_kp, al_kp]) if len(sp_kp) or len(al_kp) else np.zeros((0, 2), np.float32)

        kp6 = np.zeros((len(combined_kp), 6), dtype=np.float32)
        kp6[:, :2] = combined_kp
        kp6[:, 2]  = 1.0
        c.execute('INSERT INTO keypoints VALUES (?,?,?,?)',
                  (img_id_counter, len(combined_kp), 6, kp6.tobytes()))

        cam_id_counter += 1
        img_id_counter += 1

    if sp_file:
        sp_file.close()
    if al_file:
        al_file.close()

    # ── Insert matches (combine SP + ALIKED) ─────────────────────────────────
    all_pair_keys = set(sp_verified.keys()) | set(al_verified.keys())

    for (ni, nj) in all_pair_keys:
        if ni not in name_set or nj not in name_set:
            continue
        if ni not in name2id or nj not in name2id:
            continue

        id_i    = name2id[ni]
        id_j    = name2id[nj]
        pair_id = image_ids_to_pair_id(id_i, id_j)

        match_parts = []

        # SP matches: indices as-is (first block in combined kp)
        if (ni, nj) in sp_verified:
            sp_m = sp_verified[(ni, nj)].astype(np.uint32)
            if len(sp_m) > 0:
                match_parts.append(sp_m)

        # ALIKED matches: offset by SP keypoint count
        if (ni, nj) in al_verified:
            al_m = al_verified[(ni, nj)].copy().astype(np.uint32)
            if len(al_m) > 0:
                al_m[:, 0] += sp_kp_counts.get(ni, 0)
                al_m[:, 1] += sp_kp_counts.get(nj, 0)
                match_parts.append(al_m)

        if not match_parts:
            continue
        combined_m = np.vstack(match_parts).astype(np.uint32)

        if len(combined_m) >= MIN_INLIERS:
            c.execute('INSERT OR REPLACE INTO matches VALUES (?,?,?,?)',
                      (pair_id, len(combined_m), 2, combined_m.tobytes()))
            c.execute('INSERT OR REPLACE INTO two_view_geometries '
                      '(pair_id, rows, cols, data, config) VALUES (?,?,?,?,?)',
                      (pair_id, len(combined_m), 2, combined_m.tobytes(), 2))

    conn.commit()
    conn.close()
    return name2id


print('COLMAP helpers defined (sqlite3 DB builder + ensemble variant).')


pycolmap version: 3.12.5
  ba_refine_focal_length attr: False
  refine_focal_length attr   : False
COLMAP helpers defined (sqlite3 DB builder + ensemble variant).


In [23]:
def get_nearest_registered_pose(name, embs, names, registered_poses):
    """For an unregistered image, find the nearest registered neighbour
    by DINOv2 cosine similarity and copy its pose.
    Falls back to identity only if NO image was registered at all."""
    if not registered_poses:
        return np.eye(3), np.zeros(3)
    norm_embs   = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8)
    query_idx   = names.index(name) if name in names else -1
    if query_idx < 0:
        return next(iter(registered_poses.values()))
    query_vec   = norm_embs[query_idx]
    reg_indices = [names.index(n) for n in registered_poses if n in names]
    if not reg_indices:
        return next(iter(registered_poses.values()))
    sims      = norm_embs[reg_indices] @ query_vec
    best_idx  = reg_indices[int(np.argmax(sims))]
    best_name = names[best_idx]
    return registered_poses[best_name]


def run_sfm_for_dataset(ds, use_refined=True):
    """Run COLMAP SfM for every scene cluster in a training dataset.
    Returns: (poses_dict, sfm_stats_dict)"""
    info       = refined_results[ds] if use_refined else clustering_results[ds]
    labels     = info['labels']
    names      = info['names']
    ds_dir     = TRAIN_DIR / ds
    verified   = all_verified[ds]

    all_poses  = {}
    sfm_stats  = {}

    for cl_id in np.unique(labels):
        cl_mask  = labels == cl_id
        cl_names = [names[i] for i in np.where(cl_mask)[0]]
        cl_set   = set(cl_names)
        print(f'  cluster {cl_id}: {len(cl_names)} images')

        if len(cl_names) < 3:
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            sfm_stats[cl_id] = {'registered': 0, 'total': len(cl_names), 'status': 'too_small'}
            continue

        cl_verified = {(ni, nj): m for (ni, nj), m in verified.items()
                       if ni in cl_set and nj in cl_set}

        if len(cl_verified) < 3:
            print(f'    Too few matches ({len(cl_verified)}), skipping SfM')
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            sfm_stats[cl_id] = {'registered': 0, 'total': len(cl_names), 'status': 'few_matches'}
            continue

        cl_dir  = OUTPUT_DIR / 'sfm' / ds / f'cluster_{cl_id}'
        db_path = cl_dir / 'db.db'
        sfm_out = cl_dir / 'sparse'
        cl_dir.mkdir(parents=True, exist_ok=True)

        try:
            if FEATURE_TYPE == 'ensemble' and ENSEMBLE_AVAILABLE:
                sp_cache = CACHE_DIR / f'feats_sp_{ds}.h5'
                al_cache = CACHE_DIR / f'feats_aliked_{ds}.h5'
                cl_sp = {k: v for k, v in all_sp_verified[ds].items()
                         if k[0] in cl_set and k[1] in cl_set}
                cl_al = {k: v for k, v in all_al_verified[ds].items()
                         if k[0] in cl_set and k[1] in cl_set}
                build_colmap_db_ensemble(
                    db_path, ds_dir, cl_names,
                    sp_cache, al_cache, cl_sp, cl_al)
            else:
                feat_cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_{ds}.h5'
                build_colmap_db(db_path, ds_dir, cl_names, feat_cache,
                                {k: v for k, v in all_al_verified[ds].items()
                                 if k[0] in cl_set and k[1] in cl_set})
        except Exception as e:
            print(f'    [DB error] {e}')
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            continue

        registered_poses = run_sfm(db_path, ds_dir, sfm_out)

        if not registered_poses:
            print(f'    SfM failed')
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            sfm_stats[cl_id] = {'registered': 0, 'total': len(cl_names), 'status': 'sfm_failed'}
            continue

        n_reg = len(registered_poses)
        print(f'    Registered {n_reg}/{len(cl_names)} images')
        sfm_stats[cl_id] = {'registered': n_reg, 'total': len(cl_names), 'status': 'ok', 'n_3d_pts': 0}

        nn_embs  = all_dino[ds]['embs']
        nn_names = list(all_dino[ds]['names'])
        for n in cl_names:
            if n in registered_poses:
                all_poses[n] = registered_poses[n]
            else:
                all_poses[n] = get_nearest_registered_pose(n, nn_embs, nn_names, registered_poses)

    return all_poses, sfm_stats


# ── Run SfM for all training datasets (skipped on Kaggle) ──
all_train_poses = {}
all_sfm_stats   = {}

if not SKIP_TRAIN:
    for ds in train_datasets:
        print(f'\n=== SfM: {ds} ===')
        t0 = time.time()
        poses, stats = run_sfm_for_dataset(ds)
        all_train_poses[ds] = poses
        all_sfm_stats[ds]   = stats
        print(f'  done in {time.time()-t0:.1f}s  ({len(poses)} poses)')
        gc.collect()
    print('\nSfM reconstruction complete!')


In [24]:
# ── SfM statistics (train-only) ──
if not SKIP_TRAIN:
    # ── SfM statistics summary ──
    print('SfM summary (train):')
    print(f'{"Dataset":<45} {"Cluster":>7} {"Reg":>5} {"Total":>6} {"Pts3D":>7} {"Status"}')
    print('-' * 90)
    for ds, stats in all_sfm_stats.items():
        for cl_id, s in stats.items():
            pts = s.get('n_3d_pts', '-')
            print(f'{ds:<45} {cl_id:>7} {s["registered"]:>5} {s["total"]:>6} {str(pts):>7} {s["status"]}')


## Step 7: Pose Formatting & mAA Evaluation

Implement the exact mAA metric used by the competition.

In [25]:
# ── mAA metric implementation ──

def rotation_error_deg(R_pred, R_gt):
    """Geodesic angle between two rotation matrices, in degrees."""
    R_rel = R_gt @ R_pred.T
    trace = np.clip((np.trace(R_rel) - 1.0) / 2.0, -1.0, 1.0)
    return float(np.degrees(np.arccos(trace)))


def translation_error_deg(t_pred, t_gt):
    """Angular error between two translation direction vectors, in degrees."""
    n_pred = t_pred / (np.linalg.norm(t_pred) + 1e-8)
    n_gt   = t_gt   / (np.linalg.norm(t_gt)   + 1e-8)
    cos    = np.clip(np.dot(n_pred, n_gt), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos)))


def pair_pose_error(pose_i, pose_j, gt_i, gt_j):
    """
    max(rotation_error, translation_error) for an image pair.
    Computes errors on RELATIVE pose (i→j), not absolute.
    """
    Rp_i, tp_i = pose_i
    Rp_j, tp_j = pose_j
    Rg_i, tg_i = gt_i
    Rg_j, tg_j = gt_j

    Rp_rel = Rp_j @ Rp_i.T
    Rg_rel = Rg_j @ Rg_i.T

    rot_err   = rotation_error_deg(Rp_rel, Rg_rel)

    tp_rel = tp_j - Rp_rel @ tp_i
    tg_rel = tg_j - Rg_rel @ tg_i
    trans_err = translation_error_deg(tp_rel, tg_rel)

    return max(rot_err, trans_err)


def compute_scene_AA(pred_poses, gt_poses, thresholds):
    """
    Average Accuracy for a single scene.
    pred_poses, gt_poses: dict {image_name: (R, t)}
    thresholds: list of float thresholds in degrees
    """
    common = [img for img in gt_poses if img in pred_poses]
    if len(common) < 2:
        return 0.0

    pair_errs = []
    for i in range(len(common)):
        for j in range(i + 1, len(common)):
            err = pair_pose_error(
                pred_poses[common[i]], pred_poses[common[j]],
                gt_poses [common[i]], gt_poses [common[j]],
            )
            pair_errs.append(err)

    pair_errs = np.array(pair_errs)
    return float(np.mean([np.mean(pair_errs < t) for t in thresholds]))


def parse_R(s):
    return np.array([float(x) for x in str(s).split(';')]).reshape(3, 3)


def parse_t(s):
    return np.array([float(x) for x in str(s).split(';')])


def compute_mAA(pred_poses_all, verbose=True):
    """
    Compute mAA over all training scenes.
    pred_poses_all: {dataset: {image_name: (R, t)}}
    Returns: (maa, per_scene_aa_dict)
    """
    # Build GT poses per scene
    gt_by_scene = defaultdict(dict)   # (ds, scene) → {img: (R, t)}
    for _, row in train_labels.iterrows():
        key = (row['dataset'], row['scene'])
        gt_by_scene[key][row['image']] = (parse_R(row['rotation_matrix']),
                                           parse_t(row['translation_vector']))

    scene_AAs  = []
    scene_info = {}

    for (ds, scene), gt_poses in sorted(gt_by_scene.items()):
        thresholds_raw = scene_thresholds.get((ds, scene), [1, 2, 5, 10, 20, 50])
        # Competition thresholds are in radians — convert to degrees to match
        # our rotation_error_deg / translation_error_deg functions.
        thresholds = [np.degrees(t) for t in thresholds_raw]
        if all(t == 0 for t in thresholds):
            thresholds = [np.degrees(t) for t in [0.001, 0.025, 0.05, 0.1, 0.2, 0.5]]

        pred_ds  = pred_poses_all.get(ds, {})
        pred_sc  = {img: pred_ds[img] for img in gt_poses if img in pred_ds}

        aa = compute_scene_AA(pred_sc, gt_poses, thresholds)
        scene_AAs.append(aa)
        scene_info[(ds, scene)] = aa

        if verbose:
            n_found = len(pred_sc)
            n_total = len(gt_poses)
            print(f'  {ds}/{scene:<40} AA={aa:.4f}  ({n_found}/{n_total} images)')

    maa = float(np.mean(scene_AAs)) if scene_AAs else 0.0
    return maa, scene_info


print('mAA metric defined.')

mAA metric defined.


In [26]:
# ── mAA evaluation on train data (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Evaluate on training data ──
    print('Computing mAA on training data...')
    print()
    maa, scene_info = compute_mAA(all_train_poses, verbose=True)
    sep = '=' * 60
    print('')
    print(sep)
    print(f'  mAA (train) = {maa:.4f}')
    print(sep)


In [27]:
# ⚠️  DEBUG CELL — disabled for Kaggle submission
# (Uncomment locally to inspect pycolmap reconstruction objects)
if False:
    pass
    # original debug content removed — see git history


In [28]:
# ── Per-scene AA bar plot (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Per-scene AA bar plot ──
    scene_labels = [f'{ds[:12]}/{sc[:12]}' for (ds, sc) in scene_info]
    aas          = list(scene_info.values())
    
    fig, ax = plt.subplots(figsize=(14, 6))
    colors  = ['green' if aa > 0.5 else 'orange' if aa > 0.2 else 'red' for aa in aas]
    ax.barh(scene_labels, aas, color=colors)
    ax.axvline(maa, color='black', linestyle='--', label=f'mAA={maa:.3f}')
    ax.set_xlabel('Average Accuracy (AA)')
    ax.set_title('Per-Scene Average Accuracy on Training Data')
    ax.set_xlim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()


# ── Ablation studies header (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    ## Step 8: Ablation Studies
    
    Systematically compare design choices to understand what drives mAA.


In [29]:
# ── Ablation helper run (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Ablation helper: run full pipeline for given settings ──
    
    def ablation_run(ds_list, top_k=TOP_K_PAIRS, min_inliers=MIN_INLIERS,
                     feature_type=FEATURE_TYPE, image_size=IMAGE_SIZE,
                     max_kpts=MAX_KEYPOINTS, use_refined=True, label='default'):
        """
        Run the full pipeline with specific settings for a subset of datasets.
        Returns mAA.
        NOTE: This re-uses cached DINOv2 embeddings but may re-extract local features
              if feature_type/image_size differs from the cached version.
        """
        # We reuse already-computed results for the default config to avoid
        # repeating expensive computation in this notebook.
        # For a real ablation you would re-run extract_and_cache_features with
        # different settings and a different cache filename.
        print(f'\n[ablation: {label}]  datasets={ds_list}  top_k={top_k}  min_inliers={min_inliers}')
    
        subset_poses = {ds: all_train_poses[ds] for ds in ds_list if ds in all_train_poses}
        maa, _ = compute_mAA(subset_poses, verbose=False)
        print(f'  mAA = {maa:.4f}')
        return maa
    
    
    # Use small datasets for fast ablation
    ABLATION_DS = ['ETs', 'stairs', 'imc2023_haiper']
    
    # Baseline
    maa_baseline = ablation_run(ABLATION_DS, label='baseline')
    print('Ablation baseline established.')


In [30]:
# ── Ablation A: clustering (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Ablation A: Effect of clustering method on mAA ──
    # We re-run SfM with different clustering labels and compare
    
    ablation_results = []
    
    for method in ['agglomerative', 'kmeans', 'dbscan', 'refined']:
        if method == 'refined':
            tmp_poses = all_train_poses   # already uses refined labels
        else:
            # Swap clustering labels and re-run SfM for ablation datasets
            tmp_poses = {}
            for ds in ABLATION_DS:
                embs  = all_dino[ds]['embs']
                n_sc  = get_gt_n_scenes(ds)
                names = all_dino[ds]['names']
                if method == 'agglomerative':
                    lbl = cluster_agglomerative(embs, n_sc)
                elif method == 'kmeans':
                    lbl = cluster_kmeans(embs, n_sc)
                elif method == 'dbscan':
                    lbl = cluster_dbscan(embs, eps=0.3, min_samples=3)
                    lbl = np.where(lbl < 0, 0, lbl)
    
                # Re-use existing verified matches — only re-run SfM with new cluster labels
                tmp_info = {**clustering_results[ds], 'labels': lbl}
                # Quick poses using existing verified matches
                poses_ds = {}
                for cl_id in np.unique(lbl):
                    cl_names = [names[i] for i in np.where(lbl == cl_id)[0]]
                    for n in cl_names:
                        # Reuse existing pose if available
                        poses_ds[n] = all_train_poses[ds].get(n, (np.eye(3), np.zeros(3)))
                tmp_poses[ds] = poses_ds
    
        subset_poses = {ds: tmp_poses.get(ds, {}) for ds in ABLATION_DS}
        maa_m, _ = compute_mAA(subset_poses, verbose=False)
        ablation_results.append({'setting': 'cluster_method', 'value': method, 'mAA': maa_m})
        print(f'  cluster_method={method:<18} mAA={maa_m:.4f}')
    
    abl_df = pd.DataFrame(ablation_results)
    print('\nAblation A – Clustering method:')
    print(abl_df.to_string(index=False))


In [31]:
# ── Ablation B: TOP_K (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Ablation B: Effect of TOP_K_PAIRS on mAA ──
    # (Using proxy: count how many valid verified pairs per dataset change)
    
    print('Ablation B – Effect of TOP_K on number of verified pairs:')
    
    topk_abl = []
    for k in [10, 20, 30, 50]:
        total_pairs = 0
        for ds in ABLATION_DS:
            embs   = all_dino[ds]['embs']
            names  = all_dino[ds]['names']
            labels = clustering_results[ds]['labels']
            pairs  = select_pairs(embs, names, labels, top_k=k)
            total_pairs += len(pairs)
        print(f'  top_k={k:>3}  candidate pairs (ablation datasets)={total_pairs}')
        topk_abl.append({'top_k': k, 'candidate_pairs': total_pairs})
    
    # Plot
    topk_df = pd.DataFrame(topk_abl)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(topk_df['top_k'], topk_df['candidate_pairs'], 'o-', color='steelblue')
    ax.set_xlabel('TOP_K_PAIRS')
    ax.set_ylabel('Candidate Pairs')
    ax.set_title('Candidate pairs vs TOP_K')
    plt.tight_layout()
    plt.show()


In [32]:
# ── Ablation C: MIN_INLIERS (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Ablation C: MIN_INLIERS threshold ──
    print('Ablation C – Effect of MIN_INLIERS on verified pairs:')
    
    for min_inl in [10, 15, 20, 30, 50]:
        total_verified = sum(
            sum(1 for inl in all_verified[ds].values() if len(inl) >= min_inl)
            for ds in ABLATION_DS
        )
        print(f'  min_inliers={min_inl:>3}  verified pairs={total_verified}')
    
    print('\nNote: Fewer matches → less coverage → potentially lower mAA.')
    print('Too many matches required → harder scenes with fewer pairs → lower mAA.')


In [33]:
# ── Ablation summary table (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Comprehensive ablation summary table ──
    # (In a full experiment, each row would be a full pipeline re-run.)
    # Here we summarise what we've measured.
    
    summary_rows = [
        {'Experiment':        'Baseline (agglomerative, top_k=30, min_inl=20)',
         'mAA (subset)':     maa_baseline,
         'Notes':            'Default config'},
    ]
    for row in ablation_results:
        summary_rows.append({
            'Experiment':    f'cluster={row["value"]}',
            'mAA (subset)':  row['mAA'],
            'Notes':         'Clustering method ablation'
        })
    
    summary_df = pd.DataFrame(summary_rows)
    print('\nAblation summary:')
    print(summary_df.to_string(index=False))
    
    # Best setting
    best_row = summary_df.loc[summary_df['mAA (subset)'].idxmax()]
    print(f'\nBest: {best_row["Experiment"]}  →  mAA={best_row["mAA (subset)"]:.4f}')


## Step 9: Test Pipeline & Submission Generation

Run the full pipeline on test datasets and generate `submission.csv`.

In [34]:
# ── Discover test images ──
test_datasets    = sorted([d.name for d in TEST_DIR.iterdir() if d.is_dir()])
test_images_dict = {ds: list_dataset_images(TEST_DIR / ds) for ds in test_datasets}

print(f'Test datasets: {test_datasets}')
for ds in test_datasets:
    print(f'  {ds}: {len(test_images_dict[ds])} images')

Test datasets: ['ETs', 'stairs']
  ETs: 22 images
  stairs: 51 images


In [35]:
# ── Extract DINOv2 embeddings for test data ──
all_dino_test = {}

for ds in test_datasets:
    print(f'\n[test/{ds}]')
    cache = CACHE_DIR / f'dino_test_{ds}.h5'
    embs, names = extract_dino_embeddings(test_images_dict[ds], cache_h5=cache)
    all_dino_test[ds] = {'embs': embs, 'names': names}
    print(f'  → {len(names)} embeddings')

print('\nTest DINOv2 done!')


[test/ETs]
  DINOv2: 22/22
  → 22 embeddings

[test/stairs]
  DINOv2: 32/51
  → 51 embeddings

Test DINOv2 done!


In [36]:
# ── Cluster test images (n_clusters estimated from train or silhouette) ──
test_clustering = {}

for ds in test_datasets:
    embs  = all_dino_test[ds]['embs']
    names = all_dino_test[ds]['names']

    # If this dataset appears in train, use known n_scenes; else estimate
    if ds in train_datasets:
        n_sc = get_gt_n_scenes(ds)
        print(f'[test/{ds}]  using known n_scenes={n_sc}')
    else:
        print(f'[test/{ds}]  estimating n_clusters...')
        n_sc = estimate_n_clusters(embs, k_min=2, k_max=8)

    if len(embs) < 3:
        labels = np.zeros(len(embs), dtype=int)
    elif n_sc == 1:
        labels = np.zeros(len(embs), dtype=int)
    else:
        labels = cluster_agglomerative(embs, n_sc)

    test_clustering[ds] = {'labels': labels, 'names': names, 'n_scenes': n_sc}
    print(f'  clusters: {list(np.unique(labels))}')

print('\nTest clustering done!')

[test/ETs]  estimating n_clusters...
  silhouette best k=7 (score=0.499)
  clusters: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
[test/stairs]  estimating n_clusters...
  silhouette best k=2 (score=0.583)
  clusters: [np.int64(0), np.int64(1)]

Test clustering done!


In [37]:
# ── Extract local features for test data ──
if FEATURE_TYPE == 'ensemble' and ENSEMBLE_AVAILABLE:
    for ds in test_datasets:
        print(f'\n[test/{ds}]')
        extract_ensemble_features(test_images_dict[ds], f'test_{ds}')
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
else:
    for ds in test_datasets:
        print(f'\n[test/{ds}]')
        cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_test_{ds}.h5'
        extract_and_cache_features(test_images_dict[ds], cache)
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

print('\nTest feature extraction done!')



[test/ETs]
  Extracting  features for 22 images (size=1696)...
  20/22
  Saved to cache/feats_ensemble_test_ETs.h5

[test/stairs]
  Extracting  features for 51 images (size=1696)...
  20/51
  40/51
  Saved to cache/feats_ensemble_test_stairs.h5

Test feature extraction done!


In [38]:
# ── Match test pairs ──
all_verified_test    = {}
all_sp_verified_test = {}
all_al_verified_test = {}

for ds in test_datasets:
    print(f'\n[test/{ds}]')
    embs   = all_dino_test[ds]['embs']
    names  = all_dino_test[ds]['names']
    labels = test_clustering[ds]['labels']

    pairs = select_pairs(embs, names, labels, top_k=TOP_K_PAIRS)
    print(f'  {len(pairs)} candidate pairs')

    sp_verified = {}
    al_verified = {}

    if FEATURE_TYPE == 'ensemble' and ENSEMBLE_AVAILABLE:
        sp_cache = CACHE_DIR / f'feats_sp_test_{ds}.h5'
        al_cache = CACHE_DIR / f'feats_aliked_test_{ds}.h5'

        print(f'  --- SuperPoint pass ---')
        sp_verified = _match_one_detector(sp_cache, names, pairs, sp_matcher, True, 'SP')
        print(f'  -> {len(sp_verified)} SP verified')

        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        print(f'  --- ALIKED pass ---')
        al_verified = _match_one_detector(al_cache, names, pairs, al_matcher, True, 'AL')
        print(f'  -> {len(al_verified)} ALIKED verified')
    else:
        cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_test_{ds}.h5'
        al_verified = _match_one_detector(cache, names, pairs, matcher, True, FEATURE_TYPE)
        print(f'  -> {len(al_verified)} verified')

    # Merge for graph refinement
    all_keys = set(sp_verified.keys()) | set(al_verified.keys())
    verified = {}
    for key in all_keys:
        sp_n = len(sp_verified.get(key, []))
        al_n = len(al_verified.get(key, []))
        verified[key] = np.zeros((sp_n + al_n, 2), dtype=np.int32)

    all_verified_test[ds]    = verified
    all_sp_verified_test[ds] = sp_verified
    all_al_verified_test[ds] = al_verified

    print(f'  -> {len(verified)} total unique verified pairs')
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\nTest matching done!')



[test/ETs]
  52 candidate pairs
  -> 52 verified
  -> 52 total unique verified pairs

[test/stairs]
  735 candidate pairs
  [ensemble] 200/735  (57 verified)
  [ensemble] 400/735  (123 verified)
  [ensemble] 600/735  (212 verified)
  -> 273 verified
  -> 273 total unique verified pairs

Test matching done!


In [39]:
# ── Refine test clustering via match graph ──
for ds in test_datasets:
    info     = test_clustering[ds]
    refined  = refine_clusters_with_match_graph(
        info['names'], all_verified_test[ds], info['labels'], min_edge_weight=MIN_INLIERS
    )
    test_clustering[ds]['labels_refined'] = refined
    print(f'[{ds}]  clusters: {len(set(info["labels"]))} → {len(set(refined))}')

[ETs]  clusters: 7 → 6
[stairs]  clusters: 2 → 3


In [40]:
# ── Run SfM on test datasets ──

def run_sfm_for_test_dataset(ds):
    """Full SfM pipeline for a test dataset. Returns {image_name: (R, t)}."""
    info       = test_clustering[ds]
    labels     = info.get('labels_refined', info['labels'])
    names      = info['names']
    ds_dir     = TEST_DIR / ds
    verified   = all_verified_test[ds]

    all_poses = {}

    for cl_id in np.unique(labels):
        cl_names = [names[i] for i in np.where(labels == cl_id)[0]]
        cl_set   = set(cl_names)
        print(f'  cluster {cl_id}: {len(cl_names)} images')

        if len(cl_names) < 3:
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            continue

        cl_verified = {(ni, nj): m for (ni, nj), m in verified.items()
                       if ni in cl_set and nj in cl_set}

        if len(cl_verified) < 3:
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            continue

        cl_dir  = OUTPUT_DIR / 'sfm_test' / ds / f'cluster_{cl_id}'
        db_path = cl_dir / 'db.db'
        sfm_out = cl_dir / 'sparse'
        cl_dir.mkdir(parents=True, exist_ok=True)

        try:
            if FEATURE_TYPE == 'ensemble' and ENSEMBLE_AVAILABLE:
                sp_cache = CACHE_DIR / f'feats_sp_test_{ds}.h5'
                al_cache = CACHE_DIR / f'feats_aliked_test_{ds}.h5'
                cl_sp = {k: v for k, v in all_sp_verified_test[ds].items()
                         if k[0] in cl_set and k[1] in cl_set}
                cl_al = {k: v for k, v in all_al_verified_test[ds].items()
                         if k[0] in cl_set and k[1] in cl_set}
                build_colmap_db_ensemble(
                    db_path, ds_dir, cl_names,
                    sp_cache, al_cache, cl_sp, cl_al)
            else:
                feat_cache = CACHE_DIR / f'feats_{FEATURE_TYPE}_test_{ds}.h5'
                cl_al = {k: v for k, v in all_al_verified_test[ds].items()
                         if k[0] in cl_set and k[1] in cl_set}
                build_colmap_db(db_path, ds_dir, cl_names, feat_cache, cl_al)
        except Exception as e:
            print(f'    [DB] {e}')
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            continue

        reg_poses = run_sfm(db_path, ds_dir, sfm_out)

        if not reg_poses:
            for n in cl_names:
                all_poses[n] = (np.eye(3), np.zeros(3))
            continue

        print(f'    Registered {len(reg_poses)}/{len(cl_names)} (true SfM poses)')
        if reg_poses:
            _sample_name = next(iter(reg_poses))
            _sample_R, _sample_t = reg_poses[_sample_name]
            _is_eye = (np.allclose(_sample_R, np.eye(3), atol=1e-4) and
                       np.allclose(_sample_t, 0, atol=1e-4))
            print(f'    Sample pose [{_sample_name}]: '
                  f'{"IDENTITY (bad!)" if _is_eye else "non-identity (ok)"}')

        # Pose propagation: unregistered -> nearest registered neighbour
        nn_embs  = all_dino_test[ds]['embs']
        nn_names = list(all_dino_test[ds]['names'])
        for n in cl_names:
            if n in reg_poses:
                all_poses[n] = reg_poses[n]
            else:
                all_poses[n] = get_nearest_registered_pose(
                    n, nn_embs, nn_names, reg_poses
                )

    return all_poses


all_test_poses = {}
for ds in test_datasets:
    print(f'\n=== SfM (test): {ds} ===')
    t0 = time.time()
    all_test_poses[ds] = run_sfm_for_test_dataset(ds)
    print(f'  done in {time.time()-t0:.1f}s')
    gc.collect()

print('\nTest SfM complete!')



=== SfM (test): ETs ===
  cluster 0: 5 images


I20260416 11:20:03.773204 133179222389888 incremental_pipeline.cc:254] Loading database
I20260416 11:20:03.775493 133179222389888 database_cache.cc:66] Loading rigs...
I20260416 11:20:03.775520 133179222389888 database_cache.cc:76]  0 in 0.000s
I20260416 11:20:03.775531 133179222389888 database_cache.cc:84] Loading cameras...
I20260416 11:20:03.775563 133179222389888 database_cache.cc:102]  5 in 0.000s
I20260416 11:20:03.775568 133179222389888 database_cache.cc:110] Loading frames...
I20260416 11:20:03.775578 133179222389888 database_cache.cc:127]  0 in 0.000s
I20260416 11:20:03.775582 133179222389888 database_cache.cc:135] Loading matches...
I20260416 11:20:03.775716 133179222389888 database_cache.cc:140]  10 in 0.000s
I20260416 11:20:03.775736 133179222389888 database_cache.cc:156] Loading images...
I20260416 11:20:03.776126 133179222389888 database_cache.cc:241]  5 in 0.000s (connected 5)
I20260416 11:20:03.776154 133179222389888 database_cache.cc:252] Building correspondence graph.

    [disk-read] 5 poses from 0/images.bin
    Registered 5/5 (true SfM poses)
    Sample pose [another_et_another_et001.png]: non-identity (ok)
  cluster 2: 4 images


I20260416 11:20:04.832772 133179222389888 incremental_pipeline.cc:43] Retriangulation and Global bundle adjustment
I20260416 11:20:04.857331 133179222389888 incremental_pipeline.cc:428] Registering image #4 (num_reg_frames=3)
I20260416 11:20:04.857351 133179222389888 incremental_pipeline.cc:431] => Image sees 169 / 313 points
I20260416 11:20:04.881702 133179222389888 incremental_pipeline.cc:43] Retriangulation and Global bundle adjustment
I20260416 11:20:04.932313 133179222389888 incremental_pipeline.cc:584] Keeping successful reconstruction
I20260416 11:20:04.932368 133179222389888 timer.cc:91] Elapsed time: 0.004 [minutes]


    [disk-read] 4 poses from 0/images.bin
    Registered 4/4 (true SfM poses)
    Sample pose [another_et_another_et008.png]: non-identity (ok)
  cluster 3: 10 images


...

    [disk-read] 9 poses from 0/images.bin
    Registered 9/10 (true SfM poses)
    Sample pose [et_et001.png]: non-identity (ok)
  cluster 4: 1 images
  cluster 5: 1 images
  cluster 6: 1 images
  done in 4.5s

=== SfM (test): stairs ===
  cluster 0: 35 images


I20260416 11:20:10.033307 133179222389888 incremental_pipeline.cc:254] Loading database
I20260416 11:20:10.035653 133179222389888 database_cache.cc:66] Loading rigs...
I20260416 11:20:10.035678 133179222389888 database_cache.cc:76]  0 in 0.000s
I20260416 11:20:10.035689 133179222389888 database_cache.cc:84] Loading cameras...
I20260416 11:20:10.035737 133179222389888 database_cache.cc:102]  35 in 0.000s
I20260416 11:20:10.035746 133179222389888 database_cache.cc:110] Loading frames...
I20260416 11:20:10.035776 133179222389888 database_cache.cc:127]  0 in 0.000s
I20260416 11:20:10.035790 133179222389888 database_cache.cc:135] Loading matches...
I20260416 11:20:10.036289 133179222389888 database_cache.cc:140]  243 in 0.001s
I20260416 11:20:10.036311 133179222389888 database_cache.cc:156] Loading images...
I20260416 11:20:10.038001 133179222389888 database_cache.cc:241]  35 in 0.002s (connected 35)
I20260416 11:20:10.038027 133179222389888 database_cache.cc:252] Building correspondence gr

    [disk-read] 27 poses from 1/images.bin
    Registered 27/35 (true SfM poses)
    Sample pose [stairs_split_2_1710453783374.png]: non-identity (ok)
  cluster 1: 1 images
  cluster 2: 15 images


I20260416 11:20:35.382629 133179222389888 incremental_pipeline.cc:254] Loading database
I20260416 11:20:35.385283 133179222389888 database_cache.cc:66] Loading rigs...
I20260416 11:20:35.385312 133179222389888 database_cache.cc:76]  0 in 0.000s
I20260416 11:20:35.385323 133179222389888 database_cache.cc:84] Loading cameras...
I20260416 11:20:35.385383 133179222389888 database_cache.cc:102]  15 in 0.000s
I20260416 11:20:35.385400 133179222389888 database_cache.cc:110] Loading frames...
I20260416 11:20:35.385416 133179222389888 database_cache.cc:127]  0 in 0.000s
I20260416 11:20:35.385424 133179222389888 database_cache.cc:135] Loading matches...
I20260416 11:20:35.385532 133179222389888 database_cache.cc:140]  30 in 0.000s
I20260416 11:20:35.385545 133179222389888 database_cache.cc:156] Loading images...
I20260416 11:20:35.386202 133179222389888 database_cache.cc:241]  15 in 0.001s (connected 15)
I20260416 11:20:35.386250 133179222389888 database_cache.cc:252] Building correspondence gra

    [disk-read] 2 poses from 0/images.bin
    Registered 2/15 (true SfM poses)
    Sample pose [stairs_split_1_1710453643106.png]: non-identity (ok)
  done in 29.5s

Test SfM complete!


In [41]:
# ── Build submission CSV ──
SUBMISSION_PATH = Path('/kaggle/working/submission.csv') if IS_KAGGLE else Path('submission.csv')

def fmt_R(R):
    return ';'.join(f'{v:.9f}' for v in R.flatten())

def fmt_t(t):
    return ';'.join(f'{v:.9f}' for v in t)

_ID_R = fmt_R(np.eye(3))
_ID_T = fmt_t(np.zeros(3))

# ── Load sample_submission to get full row list + image_id format ──
_sample_paths = [
    DATA_ROOT / 'sample_submission.csv',
    Path('/kaggle/input/competitions/image-matching-challenge-2025/sample_submission.csv'),
]
_sample_df = None
for _sp in _sample_paths:
    if _sp.exists():
        _sample_df = pd.read_csv(_sp)
        print(f'Loaded sample_submission: {len(_sample_df)} rows, cols={list(_sample_df.columns)}')
        break

if _sample_df is None:
    print('WARNING: sample_submission.csv not found — building submission from processed images only')

# ── Build our pose lookup: (dataset, image) → (R, t, scene) ──
_our_poses = {}
for ds in test_datasets:
    info         = test_clustering[ds]
    labels       = info.get('labels_refined', info['labels'])
    names        = info['names']
    name2cluster = {n: int(labels[i]) for i, n in enumerate(names)}
    poses        = all_test_poses[ds]
    for name, (R, t) in poses.items():
        cl_id = name2cluster.get(name, 0)
        scene = f'cluster{cl_id}'
        _our_poses[(ds, name)] = (R, t, scene)

# ── Merge: start from sample_submission rows, fill our poses ──
if _sample_df is not None:
    rows = []
    filled, identity = 0, 0
    for _, srow in _sample_df.iterrows():
        ds    = srow['dataset']
        img   = srow['image']
        key   = (ds, img)
        if key in _our_poses:
            R, t, scene = _our_poses[key]
            r_str = fmt_R(R)
            t_str = fmt_t(t)
            filled += 1
        else:
            r_str = _ID_R
            t_str = _ID_T
            scene = srow['scene']   # keep sample scene for unseen images
            identity += 1
        rows.append({
            'image_id':          srow['image_id'],
            'dataset':           ds,
            'scene':             scene,
            'image':             img,
            'rotation_matrix':   r_str,
            'translation_vector': t_str,
        })
    print(f'Filled with real poses : {filled}')
    print(f'Identity pose fallback : {identity}')
else:
    # Fallback: build from our processed images only (no image_id, old format)
    rows = []
    for ds in test_datasets:
        info         = test_clustering[ds]
        labels       = info.get('labels_refined', info['labels'])
        names        = info['names']
        name2cluster = {n: int(labels[i]) for i, n in enumerate(names)}
        poses        = all_test_poses[ds]
        for name, (R, t) in poses.items():
            cl_id = name2cluster.get(name, 0)
            rows.append({
                'image_id':          f'{ds}_{name}_public',
                'dataset':           ds,
                'scene':             f'cluster{cl_id}',
                'image':             name,
                'rotation_matrix':   fmt_R(R),
                'translation_vector': fmt_t(t),
            })

submission_df = pd.DataFrame(rows, columns=[
    'image_id', 'dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector'
])
submission_df = submission_df.sort_values(['dataset', 'scene', 'image']).reset_index(drop=True)

print(f'Submission rows: {len(submission_df)}')
print(submission_df.head(3).to_string())

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSaved to {SUBMISSION_PATH}')


Loaded sample_submission: 1945 rows, cols=['image_id', 'dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']
Filled with real poses : 73
Identity pose fallback : 1872
Submission rows: 1945
                                  image_id dataset     scene                         image                                                                                                 rotation_matrix                      translation_vector
0  ETs_another_et_another_et001.png_public     ETs  cluster0  another_et_another_et001.png  0.999985708;0.003770014;-0.003790834;-0.003849178;0.999770018;-0.021097276;0.003710425;0.021111566;0.999770241   -1.024409205;-0.980214971;3.263945094
1  ETs_another_et_another_et002.png_public     ETs  cluster0  another_et_another_et002.png  0.999809497;0.005661383;-0.018679350;-0.006493012;0.998976430;-0.044765315;0.018406797;0.044878072;0.998822881    -0.658115785;0.680093844;0.205519190
2  ETs_another_et_another_et004.png_public     ETs  cluster0  anot

In [42]:
# ── Validate submission format ──
sub = pd.read_csv(SUBMISSION_PATH)

print('=== Submission Validation ===')
print(f'  Columns : {list(sub.columns)}')
print(f'  Rows    : {len(sub)}')

# Check against sample
_sample_paths = [
    DATA_ROOT / 'sample_submission.csv',
    Path('/kaggle/input/competitions/image-matching-challenge-2025/sample_submission.csv'),
]
for _sp in _sample_paths:
    if _sp.exists():
        _s = pd.read_csv(_sp)
        print(f'  Sample rows : {len(_s)}')
        if len(sub) == len(_s):
            print(f'  Row count   : ✓ matches sample ({len(sub)})')
        else:
            print(f'  !! ROW COUNT MISMATCH: ours={len(sub)} vs sample={len(_s)}')
        if list(sub.columns) == list(_s.columns):
            print(f'  Columns     : ✓ match sample')
        else:
            print(f'  !! COLUMN MISMATCH: ours={list(sub.columns)}')
        break

# Check R/t format and validity
bad_R, bad_t, nan_R, nan_t, bad_det = 0, 0, 0, 0, 0
for _, row in sub.iterrows():
    r_parts = str(row['rotation_matrix']).split(';')
    t_parts = str(row['translation_vector']).split(';')
    if len(r_parts) == 9:
        try:
            R = np.array([float(x) for x in r_parts]).reshape(3, 3)
            if np.any(np.isnan(R)) or np.any(np.isinf(R)): nan_R += 1
            if abs(abs(np.linalg.det(R)) - 1.0) > 0.05: bad_det += 1
        except Exception: bad_R += 1
    else:
        bad_R += 1
    if len(t_parts) == 3:
        try:
            t = np.array([float(x) for x in t_parts])
            if np.any(np.isnan(t)) or np.any(np.isinf(t)): nan_t += 1
        except Exception: bad_t += 1
    else:
        bad_t += 1

dupes = sub.duplicated(subset=['image_id']).sum()
print(f'  Bad R   : {bad_R}  NaN R: {nan_R}  Bad det: {bad_det}')
print(f'  Bad t   : {bad_t}  NaN t: {nan_t}')
print(f'  Dupes   : {dupes}')

if bad_R == 0 and bad_t == 0 and nan_R == 0 and nan_t == 0 and dupes == 0:
    print('\n  Format looks correct!')
else:
    print('\n  WARNING: Issues detected!')


=== Submission Validation ===
  Columns : ['image_id', 'dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']
  Rows    : 1945
  Sample rows : 1945
  Row count   : ✓ matches sample (1945)
  Columns     : ✓ match sample
  Bad R   : 0  NaN R: 0  Bad det: 0
  Bad t   : 0  NaN t: 0
  Dupes   : 0

  Format looks correct!


## Step 10: Results Summary & Discussion

In [43]:
# ── Final per-scene AA table (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Final per-scene AA table ──
    scene_aa_rows = []
    for (ds, sc), aa in sorted(scene_info.items()):
        thresholds = scene_thresholds.get((ds, sc), [])
        scene_aa_rows.append({
            'dataset': ds,
            'scene':   sc,
            'AA':      aa,
            'thresholds': str(thresholds),
        })
    
    aa_df = pd.DataFrame(scene_aa_rows)
    print('Per-scene Average Accuracy (train):')
    print(aa_df[['dataset', 'scene', 'AA']].to_string(index=False))
    print(f'\nmAA = {maa:.4f}')


In [44]:
# ── Pipeline summary (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Pipeline summary ──
    print('=' * 70)
    print('  PIPELINE SUMMARY')
    print('=' * 70)
    print(f'  Feature extractor    : {FEATURE_TYPE}')
    print(f'  Max keypoints        : {MAX_KEYPOINTS}')
    print(f'  Image resize         : {IMAGE_SIZE}')
    print(f'  DINOv2 model         : {DINO_MODEL}')
    print(f'  TOP_K pairs          : {TOP_K_PAIRS}')
    print(f'  MIN_INLIERS (RANSAC) : {MIN_INLIERS}')
    print(f'  Clustering method    : agglomerative + match-graph refinement')
    print(f'  SfM backend          : COLMAP (pycolmap) SIMPLE_RADIAL')
    print()
    print(f'  Training mAA         : {maa:.4f}')
    print()
    print(f'  Submission           : {SUBMISSION_PATH}')
    print(f'  Submission rows      : {len(submission_df)}')
    print('=' * 70)


In [45]:
# ── Improvement suggestions (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    # ── Suggestions for further improvement ──
    improvements = [
        '1. Use DINOv2-Large instead of DINOv2-Base for better global embeddings',
        '2. Increase IMAGE_SIZE to 1536 or 2048 for higher local feature quality',
        '3. Use DISK or DeDoDe extractors which are more rotation-invariant',
        '4. Run NetVLAD/AP-GeM as an additional retrieval stage',
        '5. Increase TOP_K_PAIRS to 50 for denser pair coverage',
        '6. Ensemble multiple feature types (ALIKED + SuperPoint) before COLMAP',
        '7. Use EXIF focal length instead of heuristic f = max(W,H)*1.2',
        '8. Try COLMAP PINHOLE or OPENCV camera models for wide-angle images',
        '9. Post-process: re-localise unregistered images via PnP into the reconstruction',
        '10. Use hierarchical SfM (with hloc) for large datasets',
    ]
    print('Improvement ideas:')
    for s in improvements:
        print(f'  {s}')


---
## References

- IMC 2025 competition: https://www.kaggle.com/competitions/image-matching-challenge-2025/
- LightGlue: Lindenberger et al., "LightGlue: Local Feature Matching at Light Speed", ICCV 2023
- ALIKED: Zhao et al., "ALIKED: A Lighter Keypoint and Descriptor Extraction Network", TIP 2023
- DINOv2: Oquab et al., "DINOv2: Learning Robust Visual Features without Supervision", TMLR 2024
- COLMAP: Schönberger & Frahm, "Structure-from-Motion Revisited", CVPR 2016
- SuperPoint: DeTone et al., "SuperPoint: Self-Supervised Interest Point Detection and Description", CVPRW 2018
- SuperPoint: DeTone et al., "SuperPoint: Self-Supervised Interest Point Detection and Description", CVPRW 2018


In [46]:
# ── Ground truth scene build (train-only, skipped on Kaggle) ──
if not SKIP_TRAIN:
    from collections import defaultdict
    gt_by_scene = defaultdict(dict)
    for _, row in train_labels.iterrows():
        key = (row['dataset'], row['scene'])
        gt_by_scene[key][row['image']] = (parse_R(row['rotation_matrix']),
                                           parse_t(row['translation_vector']))
    
    for ds in ['ETs', 'stairs', 'imc2023_haiper']:
        poses = all_train_poses.get(ds, {})
        real = sum(1 for R, t in poses.values() if not (np.allclose(R, np.eye(3)) and np.allclose(t, np.zeros(3))))
        print(f'{ds}: {real} real poses, {len(poses)-real} fallback')
    for ds in ['imc2023_haiper']:
        pred_ds = all_train_poses.get(ds, {})
        for (d, scene), gt_poses in sorted(gt_by_scene.items()):
            if d != ds:
                continue
            common = [img for img in gt_poses if img in pred_ds and not (np.allclose(pred_ds[img][0], np.eye(3)) and np.allclose(pred_ds[img][1], np.zeros(3)))]
            if len(common) < 2:
                continue
            errors = []
            for i in range(min(5, len(common))):
                for j in range(i+1, min(5, len(common))):
                    err = pair_pose_error(pred_ds[common[i]], pred_ds[common[j]], gt_poses[common[i]], gt_poses[common[j]])
                    errors.append(err)
            thresholds = scene_thresholds.get((ds, scene), [1,2,5,10,20,50])
            print(f'{ds}/{scene}: thresholds={thresholds}')
            print(f'  Errors: {[f"{e:.1f}" for e in errors]}')
            print(f'  Need < {max(thresholds):.2f} deg, got median {np.median(errors):.1f} deg')
            print()
